# NATCARB v1502 — Geological Storage Data Exploration and GeoPackage Development

## Objective

This notebook explores the **National Carbon Sequestration Database and Geographic Information System (NATCARB) AllData v1502** geodatabase published by the U.S. Department of Energy (DOE) National Energy Technology Laboratory (NETL).

The immediate objectives are to:

1. inspect the structure and contents of the NATCARB v1502 File Geodatabase;
2. inventory all spatial and non-spatial layers;
3. identify layers relevant to geological CO₂ storage;
4. inspect associated source metadata and field definitions;
5. assess geometry types, coordinate reference systems, spatial coverage, attribute completeness, and Canadian records;
6. identify relationships among storage formations, basins, assessment units, wells, sources, and other NATCARB entities where applicable;
7. distinguish source fields from fields that may later be standardized or derived;
8. evaluate which NATCARB information should be retained in a cleaned research-oriented representation; and
9. develop a standardized GeoPackage representation suitable for later integration with Canadian geological storage datasets.

The notebook is exploratory. No source fields, geometries, or records should be discarded until their meaning and potential analytical value have been assessed.

---

## Source

**Dataset:** NATCARB AllData v1502  
**Publisher:** U.S. Department of Energy (DOE), National Energy Technology Laboratory (NETL)  
**Version:** v1502  
**Dataset page:**  
https://edx.netl.doe.gov/dataset/natcarb-alldata-v1502

**Download resource:**  
https://edx.netl.doe.gov/resource/f7b936b3-b250-47e4-8475-e1c8474d9e22/download

The downloaded archive contains two principal directories:

```text
NATCARB/
├── Metadata_v1502/
└── NATCARB_v1502.gdb/

## Supplemental NATCARB material

Previous project work used an updated NATCARB saline 10 km grid product released
in April 2022.

This product is treated separately from the original NATCARB v1502 geodatabase
because it represents a later augmentation of the 2015 NATCARB saline dataset.

The update preserves the original NATCARB fields and adds additional geological
information derived from more recent literature and datasets.

Of particular relevance:

- `VOL_LOW` = P10 storage resource estimate per cell
- `VOL_MED` = P50 storage resource estimate per cell
- `VOL_HIGH` = P90 storage resource estimate per cell
- `UID` links saline 10 km cells to the corresponding saline polygon formation
- `New_ID` uniquely identifies updated 10 km grid cells

The supplemental dataset will therefore be used as:

1. a reference for interpreting the original v1502 schema;
2. a comparison dataset for assessing later improvements to NATCARB;
3. a provenance source for understanding previous P50 storage-capacity analyses; and
4. a potential additional source for the eventual standardized geological-storage
   database.

It will not be silently substituted for the original NATCARB v1502 bronze data.

In [1]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyogrio

from pyproj import CRS

In [2]:
# ---------------------------------------------------------------------------
# NATCARB source paths
# ---------------------------------------------------------------------------

NATCARB_ROOT = Path(
    r"C:\Users\aviga\Research\potential data\Storage\NATCARB"
)

GDB_PATH = NATCARB_ROOT / "NATCARB_v1502.gdb"
METADATA_DIR = NATCARB_ROOT / "Metadata_v1502"

# Canadian working CRS used later for spatial standardization
CANADA_CRS = CRS.from_epsg(3347)

# ---------------------------------------------------------------------------
# Validate source directories
# ---------------------------------------------------------------------------

required_paths = {
    "NATCARB root": NATCARB_ROOT,
    "NATCARB geodatabase": GDB_PATH,
    "NATCARB metadata directory": METADATA_DIR,
}

for label, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:<28} [{status}]  {path}")

if not GDB_PATH.is_dir():
    raise FileNotFoundError(
        f"NATCARB File Geodatabase not found: {GDB_PATH}"
    )

if not METADATA_DIR.is_dir():
    raise FileNotFoundError(
        f"NATCARB metadata directory not found: {METADATA_DIR}"
    )

NATCARB root                 [FOUND]  C:\Users\aviga\Research\potential data\Storage\NATCARB
NATCARB geodatabase          [FOUND]  C:\Users\aviga\Research\potential data\Storage\NATCARB\NATCARB_v1502.gdb
NATCARB metadata directory   [FOUND]  C:\Users\aviga\Research\potential data\Storage\NATCARB\Metadata_v1502


In [3]:
# ---------------------------------------------------------------------------
# Inventory NATCARB geodatabase layers
# ---------------------------------------------------------------------------

layers = pyogrio.list_layers(GDB_PATH)

layer_inventory = pd.DataFrame(
    layers,
    columns=["layer", "geometry_type"]
)

layer_inventory["is_spatial"] = (
    layer_inventory["geometry_type"]
    .notna()
)

print(f"Total layers/tables: {len(layer_inventory):,}")
print(
    f"Spatial layers:      {layer_inventory['is_spatial'].sum():,}"
)
print(
    f"Non-spatial tables:  {(~layer_inventory['is_spatial']).sum():,}"
)

layer_inventory

Total layers/tables: 18
Spatial layers:      8
Non-spatial tables:  10


,layer,geometry_type,is_spatial
0,Domain_State,NaN,False
1,Domain_Fuel,NaN,False
2,Domain_Overlap,NaN,False
3,Domain_Duplicate,NaN,False
4,Domain_ARRA,NaN,False
5,Domain_Source_Types,NaN,False
6,Domain_Med_Calced,NaN,False
7,NATCARB_Coal_10K_v1502,MultiPolygon,True
8,Domain_Partnership,NaN,False
9,NATCARB_Coal_Poly_v1502,MultiPolygon,True


In [4]:
# ---------------------------------------------------------------------------
# Inspect NATCARB saline layer schemas
# ---------------------------------------------------------------------------

saline_layers = [
    "NATCARB_Saline_10K_v1502",
    "NATCARB_Saline_Poly_v1502",
]

for layer in saline_layers:
    info = pyogrio.read_info(
        GDB_PATH,
        layer=layer,
    )

    print("=" * 80)
    print(layer)
    print("=" * 80)
    print(f"Features:      {info['features']:,}")
    print(f"Geometry type: {info['geometry_type']}")
    print(f"CRS:           {info['crs']}")
    print("\nFields:")

    for field, dtype in zip(
        info["fields"],
        info["dtypes"],
    ):
        print(f"  {field:<30} {dtype}")

    print()

NATCARB_Saline_10K_v1502
Features:      186,675
Geometry type: MultiPolygon
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",45],PARAMETER["longitude_of_center",-100],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Fields:
  COL_ROW                        object
  PARTNERSHIP                    object
  ARRA_PROJECT                   object
  RESOURCE_NAME                  object
  BASIN_NAME                     object
  RSC_AREA_CELL                  float64
  VOL_LOW                        float64
  VOL_MED                        float64
  VOL_HIGH                       float64
  DEPTH_FT                       int32
  THICKNESS_FT  

In [5]:
# ---------------------------------------------------------------------------
# Inspect NATCARB domain / lookup tables
# ---------------------------------------------------------------------------

domain_layers = (
    layer_inventory.loc[
        ~layer_inventory["is_spatial"],
        "layer",
    ]
    .tolist()
)

domain_tables = {}

for layer in domain_layers:
    df = pyogrio.read_dataframe(
        GDB_PATH,
        layer=layer,
    )

    domain_tables[layer] = df

    print("=" * 80)
    print(layer)
    print("=" * 80)
    print(f"Rows: {len(df):,}")
    print(f"Columns: {list(df.columns)}")
    print()
    print(df.to_string(index=False))
    print()

Domain_State
Rows: 63
Columns: ['FREQUENCY', 'ST_PROV_NAME', 'STATE_PROV_ABBR']

 FREQUENCY          ST_PROV_NAME STATE_PROV_ABBR
      59.0               ALABAMA              AL
      49.0                ALASKA              AK
     302.0               ALBERTA              AB
      71.0               ARIZONA              AZ
      30.0              ARKANSAS              AR
      52.0      BRITISH COLUMBIA              BC
     182.0            CALIFORNIA              CA
      63.0              COLORADO              CO
      63.0           CONNECTICUT              CT
      16.0              DELAWARE              DE
       5.0  DISTRICT OF COLUMBIA              DC
     108.0               FLORIDA              FL
      64.0               GEORGIA              GA
      52.0                HAWAII              HI
      18.0                 IDAHO              ID
     138.0              ILLINOIS              IL
      92.0               INDIANA              IN
      63.0                  IOWA     

In [6]:
# ---------------------------------------------------------------------------
# NATCARB saline 10 km attribute reconnaissance
# ---------------------------------------------------------------------------

saline_10k = pyogrio.read_dataframe(
    GDB_PATH,
    layer="NATCARB_Saline_10K_v1502",
    read_geometry=False,
)

print(f"Rows:    {len(saline_10k):,}")
print(f"Columns: {len(saline_10k.columns):,}")

saline_10k.head()

Rows:    186,675
Columns: 23


,COL_ROW,PARTNERSHIP,ARRA_PROJECT,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,VOL_LOW,VOL_MED,VOL_HIGH,DEPTH_FT,...,TEMPERATURE_F,POROSITY_PCT,PERMEABILITY_mD,ASSESSED,CYCLE_OF_LAST_UPDATE,OVERLAP,DUPLICATE,MED_CALCED,Shape_Length,Shape_Area
0,314 - 367,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,290.0,0.037079,34.000000,1,"Atlas V, v1",1,0,0,40000.0,100000000.0
1,314 - 366,PCOR,NaN,Basal Cambrian,NaN,83000000.0,0.0,0.0,0.0,NaN,...,294.0,0.035540,15.666667,1,"Atlas V, v1",1,0,0,40000.0,100000000.0
2,314 - 365,PCOR,NaN,Basal Cambrian,NaN,93000000.0,0.0,0.0,0.0,NaN,...,299.0,0.035390,16.933332,1,"Atlas V, v1",1,0,0,40000.0,100000000.0
3,314 - 364,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,302.0,0.035677,39.333332,1,"Atlas V, v1",1,0,0,40000.0,100000000.0
4,315 - 367,PCOR,NaN,Basal Cambrian,NaN,79000000.0,0.0,0.0,0.0,NaN,...,285.0,0.040356,18.111111,1,"Atlas V, v1",1,0,0,40000.0,100000000.0


In [7]:
# ---------------------------------------------------------------------------
# QA flags and provenance summary
# ---------------------------------------------------------------------------

summary_columns = [
    "PARTNERSHIP",
    "ASSESSED",
    "OVERLAP",
    "DUPLICATE",
    "MED_CALCED",
]

for column in summary_columns:
    print("=" * 80)
    print(column)
    print("=" * 80)

    counts = (
        saline_10k[column]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

    counts["share_pct"] = (
        counts["count"] / len(saline_10k) * 100
    ).round(2)

    print(counts.to_string())
    print()

PARTNERSHIP
             count  share_pct
PARTNERSHIP                  
SWP          46417      24.87
BSCSP        43873      23.50
PCOR         29938      16.04
SECARB       22602      12.11
MRCSP        20312      10.88
WESTCARB     19039      10.20
MGSC          4494       2.41

ASSESSED
           count  share_pct
ASSESSED                   
1         170449      91.31
0          16226       8.69

OVERLAP
          count  share_pct
OVERLAP                   
0        152165      81.51
1         34510      18.49

DUPLICATE
            count  share_pct
DUPLICATE                   
0          184157      98.65
1            2518       1.35

MED_CALCED
             count  share_pct
MED_CALCED                   
0           157730      84.49
1            28945      15.51



In [8]:
# ---------------------------------------------------------------------------
# Validate storage-volume logic against NATCARB QA flags
# ---------------------------------------------------------------------------

volume_fields = [
    "VOL_LOW",
    "VOL_MED",
    "VOL_HIGH",
]

qa_checks = pd.DataFrame(
    {
        "rows": [
            len(saline_10k),
            (saline_10k["ASSESSED"] == 1).sum(),
            (saline_10k["ASSESSED"] == 0).sum(),
            (saline_10k["DUPLICATE"] == 1).sum(),
            (saline_10k["OVERLAP"] == 1).sum(),
            (
                (saline_10k["DUPLICATE"] == 1)
                & (saline_10k["OVERLAP"] == 1)
            ).sum(),
        ]
    },
    index=[
        "all_cells",
        "assessed_cells",
        "unassessed_cells",
        "duplicate_cells",
        "overlap_cells",
        "duplicate_and_overlap_cells",
    ],
)

qa_checks["share_pct"] = (
    qa_checks["rows"]
    / len(saline_10k)
    * 100
).round(2)

qa_checks

,rows,share_pct
all_cells,186675,100.00
assessed_cells,170449,91.31
unassessed_cells,16226,8.69
duplicate_cells,2518,1.35
overlap_cells,34510,18.49
duplicate_and_overlap_cells,2518,1.35


In [9]:
# ---------------------------------------------------------------------------
# Check consistency of assessment status and storage estimates
# ---------------------------------------------------------------------------

assessment_volume_check = (
    saline_10k
    .assign(
        has_low=saline_10k["VOL_LOW"].notna(),
        has_med=saline_10k["VOL_MED"].notna(),
        has_high=saline_10k["VOL_HIGH"].notna(),
    )
    .groupby("ASSESSED")[
        ["has_low", "has_med", "has_high"]
    ]
    .agg(["sum", "mean"])
)

assessment_volume_check

has_low      has_med      has_high     
             sum mean     sum mean      sum mean
ASSESSED                                        
0              0  0.0       0  0.0        0  0.0
1         170449  1.0  170449  1.0   170449  1.0

In [10]:
# ---------------------------------------------------------------------------
# Derive exploratory NATCARB QA / usability flags
# ---------------------------------------------------------------------------

saline_10k_explore = saline_10k.copy()

saline_10k_explore["has_storage_estimate"] = (
    saline_10k_explore["ASSESSED"].eq(1)
)

saline_10k_explore["exclude_duplicate"] = (
    saline_10k_explore["DUPLICATE"].eq(1)
)

saline_10k_explore["has_overlap_warning"] = (
    saline_10k_explore["OVERLAP"].eq(1)
)

saline_10k_explore["p50_method"] = (
    saline_10k_explore["MED_CALCED"]
    .map(
        {
            0: "partnership_provided",
            1: "natcarb_calculated",
        }
    )
)

# Quantitatively usable before resolving overlap logic
saline_10k_explore["usable_pre_overlap"] = (
    saline_10k_explore["has_storage_estimate"]
    & ~saline_10k_explore["exclude_duplicate"]
)

qa_summary = pd.Series(
    {
        "all_cells": len(saline_10k_explore),
        "assessed": saline_10k_explore["has_storage_estimate"].sum(),
        "duplicates_excluded": saline_10k_explore["exclude_duplicate"].sum(),
        "overlap_warning": saline_10k_explore["has_overlap_warning"].sum(),
        "overlap_nonduplicate": (
            saline_10k_explore["has_overlap_warning"]
            & ~saline_10k_explore["exclude_duplicate"]
        ).sum(),
        "usable_pre_overlap": saline_10k_explore["usable_pre_overlap"].sum(),
    },
    name="cells",
).to_frame()

qa_summary["share_pct"] = (
    qa_summary["cells"]
    / len(saline_10k_explore)
    * 100
).round(2)

qa_summary

,cells,share_pct
all_cells,186675,100.00
assessed,170449,91.31
duplicates_excluded,2518,1.35
overlap_warning,34510,18.49
overlap_nonduplicate,31992,17.14
usable_pre_overlap,168035,90.01


## Metadata-derived interpretation

The NATCARB v1502 storage layers are regional-scale screening datasets compiled
from multiple Regional Carbon Sequestration Partnership datasets.

For saline storage:

- regional datasets were converted to a common 10 km × 10 km vector grid;
- the dataset covers the USA and parts of Canada, including onshore and offshore areas;
- storage-resource estimates represent physically accessible pore volume under an
  open-system assumption;
- economic and regulatory constraints are not included;
- the data are intended for regional and national assessment rather than
  site-specific storage characterization;
- null values represent unavailable information, while zero may represent a valid
  reported value;
- not all source partnerships provided P10, P50, and P90 estimates;
- where only P50 was provided, NATCARB copied that estimate into P10 and P90;
- where P10 and P90 were provided without P50, NATCARB calculated P50 using the
  natural-log mean of the low and high estimates.

These provenance and uncertainty distinctions should be retained in any cleaned
or standardized GeoPackage representation.

In [11]:
# ---------------------------------------------------------------------------
# Identify potentially collapsed uncertainty ranges
# ---------------------------------------------------------------------------

same_low_med_high = (
    saline_10k["VOL_LOW"].notna()
    & saline_10k["VOL_LOW"].eq(saline_10k["VOL_MED"])
    & saline_10k["VOL_MED"].eq(saline_10k["VOL_HIGH"])
)

collapsed_uncertainty_summary = pd.Series(
    {
        "assessed_cells": saline_10k["ASSESSED"].eq(1).sum(),
        "low_equals_med_equals_high": same_low_med_high.sum(),
        "med_calculated": saline_10k["MED_CALCED"].eq(1).sum(),
    },
    name="cells",
).to_frame()

collapsed_uncertainty_summary["share_of_assessed_pct"] = (
    collapsed_uncertainty_summary["cells"]
    / saline_10k["ASSESSED"].eq(1).sum()
    * 100
).round(2)

collapsed_uncertainty_summary

,cells,share_of_assessed_pct
assessed_cells,170449,100.00
low_equals_med_equals_high,27381,16.06
med_calculated,28945,16.98


In [12]:
# ---------------------------------------------------------------------------
# Cross-check P50 calculation method against collapsed uncertainty ranges
# ---------------------------------------------------------------------------

uncertainty_crosscheck = pd.crosstab(
    saline_10k["MED_CALCED"],
    same_low_med_high,
    rownames=["MED_CALCED"],
    colnames=["LOW_EQ_MED_EQ_HIGH"],
    margins=True,
)

uncertainty_crosscheck

LOW_EQ_MED_EQ_HIGH,False,True,All
MED_CALCED,,,
0,130349,27381,157730
1,28945,0,28945
All,159294,27381,186675


In [13]:
# ---------------------------------------------------------------------------
# Derive storage-estimate provenance classes
# ---------------------------------------------------------------------------

saline_10k_explore["estimate_provenance"] = "other"

saline_10k_explore.loc[
    saline_10k_explore["ASSESSED"].eq(0),
    "estimate_provenance"
] = "unassessed"

saline_10k_explore.loc[
    saline_10k_explore["ASSESSED"].eq(1)
    & saline_10k_explore["MED_CALCED"].eq(1),
    "estimate_provenance"
] = "p50_calculated_from_low_high"

saline_10k_explore.loc[
    saline_10k_explore["ASSESSED"].eq(1)
    & saline_10k_explore["VOL_LOW"].eq(saline_10k_explore["VOL_MED"])
    & saline_10k_explore["VOL_MED"].eq(saline_10k_explore["VOL_HIGH"]),
    "estimate_provenance"
] = "single_estimate_copied_to_range"

saline_10k_explore.loc[
    saline_10k_explore["ASSESSED"].eq(1)
    & saline_10k_explore["MED_CALCED"].eq(0)
    & ~(
        saline_10k_explore["VOL_LOW"].eq(saline_10k_explore["VOL_MED"])
        & saline_10k_explore["VOL_MED"].eq(saline_10k_explore["VOL_HIGH"])
    ),
    "estimate_provenance"
] = "partnership_range_provided"

estimate_provenance_summary = (
    saline_10k_explore["estimate_provenance"]
    .value_counts()
    .rename("cells")
    .to_frame()
)

estimate_provenance_summary["share_pct"] = (
    estimate_provenance_summary["cells"]
    / len(saline_10k_explore)
    * 100
).round(2)

estimate_provenance_summary

,cells,share_pct
estimate_provenance,,
partnership_range_provided,114123,61.13
p50_calculated_from_low_high,28945,15.51
single_estimate_copied_to_range,27381,14.67
unassessed,16226,8.69


In [14]:
# ---------------------------------------------------------------------------
# Load NATCARB saline 10 km geometry
# ---------------------------------------------------------------------------

saline_10k_gdf = pyogrio.read_dataframe(
    GDB_PATH,
    layer="NATCARB_Saline_10K_v1502",
)

print(f"Features: {len(saline_10k_gdf):,}")
print(f"CRS:      {saline_10k_gdf.crs}")
print(f"Bounds:   {saline_10k_gdf.total_bounds}")

saline_10k_gdf.head()

Features: 186,675
CRS:      PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",45],PARAMETER["longitude_of_center",-100],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Bounds:   [-4061251.74 -2121064.32  2238748.26  4088935.68]


,COL_ROW,PARTNERSHIP,ARRA_PROJECT,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,VOL_LOW,VOL_MED,VOL_HIGH,DEPTH_FT,...,POROSITY_PCT,PERMEABILITY_mD,ASSESSED,CYCLE_OF_LAST_UPDATE,OVERLAP,DUPLICATE,MED_CALCED,Shape_Length,Shape_Area,geometry
0,314 - 367,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,0.037079,34.000000,1,"Atlas V, v1",1,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1218935.68, -12712..."
1,314 - 366,PCOR,NaN,Basal Cambrian,NaN,83000000.0,0.0,0.0,0.0,NaN,...,0.035540,15.666667,1,"Atlas V, v1",1,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1228935.68, -12712..."
2,314 - 365,PCOR,NaN,Basal Cambrian,NaN,93000000.0,0.0,0.0,0.0,NaN,...,0.035390,16.933332,1,"Atlas V, v1",1,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1238935.68, -12712..."
3,314 - 364,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,0.035677,39.333332,1,"Atlas V, v1",1,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1248935.68, -12712..."
4,315 - 367,PCOR,NaN,Basal Cambrian,NaN,79000000.0,0.0,0.0,0.0,NaN,...,0.040356,18.111111,1,"Atlas V, v1",1,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1261251.74 1218935.68, -12612..."


In [15]:
# ---------------------------------------------------------------------------
# Geometry QA
# ---------------------------------------------------------------------------

geometry_qa = pd.Series(
    {
        "features": len(saline_10k_gdf),
        "missing_geometry": saline_10k_gdf.geometry.isna().sum(),
        "empty_geometry": saline_10k_gdf.geometry.is_empty.sum(),
        "invalid_geometry": (~saline_10k_gdf.geometry.is_valid).sum(),
    },
    name="count",
).to_frame()

geometry_qa

,count
features,186675
missing_geometry,0
empty_geometry,0
invalid_geometry,0


In [16]:
# ---------------------------------------------------------------------------
# Load Canadian province and territory boundaries
# ---------------------------------------------------------------------------

CANADA_BOUNDARY_PATH = Path(
    r"C:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace"
) / "data_files" / "raw" / "basemaps" / "lpr_000b21a_e.shp"

provinces = gpd.read_file(CANADA_BOUNDARY_PATH)

print(f"Province features: {len(provinces):,}")
print(f"CRS:               {provinces.crs}")

provinces.head()

Province features: 13
CRS:               EPSG:3347


,PRUID,DGUID,PRNAME,PRENAME,PRFNAME,PREABBR,PRFABBR,LANDAREA,geometry
0,10,2021A000210,Newfoundland and Labrador / Terre-Neuve-et-Lab...,Newfoundland and Labrador,Terre-Neuve-et-Labrador,N.L.,T.-N.-L.,3.581704e+05,"MULTIPOLYGON (((8841194.729 2213093.663, 88411..."
1,11,2021A000211,Prince Edward Island / Île-du-Prince-Édouard,Prince Edward Island,Île-du-Prince-Édouard,P.E.I.,Î.-P.-É.,5.681179e+03,"MULTIPOLYGON (((8374335.443 1629502.597, 83743..."
2,12,2021A000212,Nova Scotia / Nouvelle-Écosse,Nova Scotia,Nouvelle-Écosse,N.S.,N.-É.,5.282471e+04,"MULTIPOLYGON (((8310463.217 1250722.263, 83104..."
3,13,2021A000213,New Brunswick / Nouveau-Brunswick,New Brunswick,Nouveau-Brunswick,N.B.,N.-B.,7.124850e+04,"MULTIPOLYGON (((7964100.72 1576822.289, 796410..."
4,24,2021A000224,Quebec / Québec,Quebec,Québec,Que.,Qc,1.298600e+06,"MULTIPOLYGON (((6948393.211 2760814.626, 69483..."


In [17]:
# ---------------------------------------------------------------------------
# Standardize province / territory names
# ---------------------------------------------------------------------------

PROVINCE_NAME_TO_CODE = {
    "Newfoundland and Labrador": "NL",
    "Prince Edward Island": "PE",
    "Nova Scotia": "NS",
    "New Brunswick": "NB",
    "Quebec": "QC",
    "Québec": "QC",
    "Ontario": "ON",
    "Manitoba": "MB",
    "Saskatchewan": "SK",
    "Alberta": "AB",
    "British Columbia": "BC",
    "Yukon": "YT",
    "Northwest Territories": "NT",
    "Nunavut": "NU",
}

province_name_column = next(
    column
    for column in ["PRENAME", "PRNAME", "province_name"]
    if column in provinces.columns
)

provinces = provinces.copy()

provinces["province_name"] = (
    provinces[province_name_column]
    .astype(str)
    .str.strip()
)

provinces["province"] = (
    provinces["province_name"]
    .map(PROVINCE_NAME_TO_CODE)
)

provinces = provinces[
    ["province", "province_name", "geometry"]
]

provinces.head()

,province,province_name,geometry
0,NL,Newfoundland and Labrador,"MULTIPOLYGON (((8841194.729 2213093.663, 88411..."
1,PE,Prince Edward Island,"MULTIPOLYGON (((8374335.443 1629502.597, 83743..."
2,NS,Nova Scotia,"MULTIPOLYGON (((8310463.217 1250722.263, 83104..."
3,NB,New Brunswick,"MULTIPOLYGON (((7964100.72 1576822.289, 796410..."
4,QC,Quebec,"MULTIPOLYGON (((6948393.211 2760814.626, 69483..."


In [18]:
# ---------------------------------------------------------------------------
# Identify NATCARB saline cells intersecting Canada using spatial index
# ---------------------------------------------------------------------------

# Reproject province boundaries to NATCARB source CRS
provinces_natcarb = provinces.to_crs(saline_10k_gdf.crs)

# Spatial join against individual province polygons
canadian_hits = gpd.sjoin(
    saline_10k_gdf,
    provinces_natcarb[
        ["province", "province_name", "geometry"]
    ],
    how="inner",
    predicate="intersects",
)

# A NATCARB cell can intersect more than one province,
# so keep the source row index for later resolution.
canadian_cell_indices = canadian_hits.index.unique()

saline_canada = (
    saline_10k_gdf.loc[canadian_cell_indices]
    .copy()
)

print(f"Canadian-intersecting cells: {len(saline_canada):,}")
print(
    f"Share of NATCARB saline cells: "
    f"{len(saline_canada) / len(saline_10k_gdf) * 100:.2f}%"
)

print(
    f"Cell-province intersections: {len(canadian_hits):,}"
)

Canadian-intersecting cells: 26,612
Share of NATCARB saline cells: 14.26%
Cell-province intersections: 27,190


In [19]:
# ---------------------------------------------------------------------------
# Check how many cells intersect multiple provinces / territories
# ---------------------------------------------------------------------------

province_matches_per_cell = (
    canadian_hits
    .groupby(level=0)
    ["province"]
    .nunique()
)

province_intersection_summary = (
    province_matches_per_cell
    .value_counts()
    .sort_index()
    .rename("cells")
    .to_frame()
)

province_intersection_summary.index.name = "province_count"

province_intersection_summary

,cells
province_count,
1,26037
2,572
3,3


In [20]:
# ---------------------------------------------------------------------------
# Assign provinces to cells with a single province intersection
# ---------------------------------------------------------------------------

# Count distinct province intersections per NATCARB cell
province_matches_per_cell = (
    canadian_hits
    .groupby(level=0)["province"]
    .nunique()
)

single_province_indices = (
    province_matches_per_cell[
        province_matches_per_cell == 1
    ]
    .index
)

ambiguous_indices = (
    province_matches_per_cell[
        province_matches_per_cell > 1
    ]
    .index
)

print(f"Single-province cells: {len(single_province_indices):,}")
print(f"Ambiguous cells:       {len(ambiguous_indices):,}")

Single-province cells: 26,037
Ambiguous cells:       575


In [21]:
# ---------------------------------------------------------------------------
# Extract direct province assignments
# ---------------------------------------------------------------------------

single_assignments = (
    canadian_hits
    .loc[single_province_indices]
    [["province", "province_name"]]
    .copy()
)

single_assignments["province_assignment_method"] = (
    "single_intersection"
)

single_assignments.head()

,province,province_name,province_assignment_method
0,AB,Alberta,single_intersection
1,AB,Alberta,single_intersection
2,AB,Alberta,single_intersection
3,AB,Alberta,single_intersection
4,AB,Alberta,single_intersection


In [22]:
# ---------------------------------------------------------------------------
# Resolve ambiguous cells by largest province intersection area
# ---------------------------------------------------------------------------

ambiguous_hits = (
    canadian_hits
    .loc[ambiguous_indices]
    [["province", "province_name", "index_right"]]
    .copy()
)

intersection_records = []

for cell_index, group in ambiguous_hits.groupby(level=0):

    cell_geometry = saline_10k_gdf.loc[
        cell_index,
        "geometry",
    ]

    for _, row in group.iterrows():

        province_geometry = provinces_natcarb.loc[
            row["index_right"],
            "geometry",
        ]

        intersection_area = (
            cell_geometry
            .intersection(province_geometry)
            .area
        )

        intersection_records.append(
            {
                "cell_index": cell_index,
                "province": row["province"],
                "province_name": row["province_name"],
                "intersection_area_m2": intersection_area,
            }
        )

ambiguous_intersections = pd.DataFrame(
    intersection_records
)

ambiguous_intersections.head()

,cell_index,province,province_name,intersection_area_m2
0,2794,SK,Saskatchewan,7.655859e+07
1,2794,AB,Alberta,1.393621e+07
2,2795,SK,Saskatchewan,7.153938e+07
3,2795,AB,Alberta,2.846062e+07
4,2796,SK,Saskatchewan,5.852048e+07


In [23]:
# ---------------------------------------------------------------------------
# Select province with largest intersection
# ---------------------------------------------------------------------------

largest_intersection = (
    ambiguous_intersections
    .sort_values(
        "intersection_area_m2",
        ascending=False,
    )
    .drop_duplicates(
        subset="cell_index",
        keep="first",
    )
    .set_index("cell_index")
)

largest_intersection[
    "province_assignment_method"
] = "largest_intersection"

largest_intersection.head()

,province,province_name,intersection_area_m2,province_assignment_method
cell_index,,,,
15468,NT,Northwest Territories,9.999999e+07,largest_intersection
51244,BC,British Columbia,9.999983e+07,largest_intersection
17218,AB,Alberta,9.999206e+07,largest_intersection
21741,AB,Alberta,9.999206e+07,largest_intersection
3024,AB,Alberta,9.999206e+07,largest_intersection


In [24]:
# ---------------------------------------------------------------------------
# Combine province assignments
# ---------------------------------------------------------------------------

province_assignments = pd.concat(
    [
        single_assignments[
            [
                "province",
                "province_name",
                "province_assignment_method",
            ]
        ],
        largest_intersection[
            [
                "province",
                "province_name",
                "province_assignment_method",
            ]
        ],
    ]
).sort_index()

print(
    f"Assigned cells: {len(province_assignments):,}"
)

province_assignments.head()

Assigned cells: 26,612


,province,province_name,province_assignment_method
0,AB,Alberta,single_intersection
1,AB,Alberta,single_intersection
2,AB,Alberta,single_intersection
3,AB,Alberta,single_intersection
4,AB,Alberta,single_intersection


In [26]:
canadian_hits.to_parquet(
    NATCARB_ROOT / "exploration_canadian_hits.parquet"
)

In [27]:
# ---------------------------------------------------------------------------
# Calculate province intersection area for all Canadian candidate cells
# ---------------------------------------------------------------------------

canadian_hits_area = canadian_hits.copy()

# Attach the matched province geometry from the existing spatial join
province_geometries = (
    provinces_natcarb
    [["geometry"]]
    .rename(columns={"geometry": "province_geometry"})
)

canadian_hits_area = (
    canadian_hits_area
    .join(
        province_geometries,
        on="index_right",
    )
)

# Exact intersection only for the already identified candidate pairs
canadian_hits_area["intersection_area_m2"] = [
    cell_geom.intersection(prov_geom).area
    for cell_geom, prov_geom in zip(
        canadian_hits_area.geometry,
        canadian_hits_area["province_geometry"],
    )
]

canadian_hits_area.head()

,COL_ROW,PARTNERSHIP,ARRA_PROJECT,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,VOL_LOW,VOL_MED,VOL_HIGH,DEPTH_FT,...,DUPLICATE,MED_CALCED,Shape_Length,Shape_Area,geometry,index_right,province,province_name,province_geometry,intersection_area_m2
0,314 - 367,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1218935.68, -12712...",8,AB,Alberta,"POLYGON ((-561673.363 1695796.522, -561783.346...",100000000.0
1,314 - 366,PCOR,NaN,Basal Cambrian,NaN,83000000.0,0.0,0.0,0.0,NaN,...,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1228935.68, -12712...",8,AB,Alberta,"POLYGON ((-561673.363 1695796.522, -561783.346...",100000000.0
2,314 - 365,PCOR,NaN,Basal Cambrian,NaN,93000000.0,0.0,0.0,0.0,NaN,...,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1238935.68, -12712...",8,AB,Alberta,"POLYGON ((-561673.363 1695796.522, -561783.346...",100000000.0
3,314 - 364,PCOR,NaN,Basal Cambrian,NaN,11000000.0,0.0,0.0,0.0,NaN,...,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1271251.74 1248935.68, -12712...",8,AB,Alberta,"POLYGON ((-561673.363 1695796.522, -561783.346...",100000000.0
4,315 - 367,PCOR,NaN,Basal Cambrian,NaN,79000000.0,0.0,0.0,0.0,NaN,...,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-1261251.74 1218935.68, -12612...",8,AB,Alberta,"POLYGON ((-561673.363 1695796.522, -561783.346...",100000000.0


In [28]:
# ---------------------------------------------------------------------------
# Calculate share of each NATCARB cell lying within Canada
# ---------------------------------------------------------------------------

canadian_area_by_cell = (
    canadian_hits_area
    .groupby(level=0)["intersection_area_m2"]
    .sum()
    .rename("canada_intersection_area_m2")
)

saline_canada = (
    saline_canada
    .join(canadian_area_by_cell)
)

saline_canada["cell_area_m2"] = (
    saline_canada.geometry.area
)

saline_canada["canada_area_fraction"] = (
    saline_canada["canada_intersection_area_m2"]
    / saline_canada["cell_area_m2"]
)

saline_canada["canada_area_pct"] = (
    saline_canada["canada_area_fraction"] * 100
)

saline_canada[
    [
        "COL_ROW",
        "RESOURCE_NAME",
        "canada_intersection_area_m2",
        "cell_area_m2",
        "canada_area_pct",
    ]
].head()

,COL_ROW,RESOURCE_NAME,canada_intersection_area_m2,cell_area_m2,canada_area_pct
0,314 - 367,Basal Cambrian,100000000.0,100000000.0,100.0
1,314 - 366,Basal Cambrian,100000000.0,100000000.0,100.0
2,314 - 365,Basal Cambrian,100000000.0,100000000.0,100.0
3,314 - 364,Basal Cambrian,100000000.0,100000000.0,100.0
4,315 - 367,Basal Cambrian,100000000.0,100000000.0,100.0


In [29]:
# ---------------------------------------------------------------------------
# Inspect Canadian overlap fractions
# ---------------------------------------------------------------------------

print(
    saline_canada["canada_area_pct"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

overlap_bins = pd.cut(
    saline_canada["canada_area_pct"],
    bins=[
        0,
        1,
        10,
        25,
        50,
        75,
        90,
        99,
        100.0001,
    ],
    right=False,
)

overlap_distribution = (
    overlap_bins
    .value_counts()
    .sort_index()
    .rename("cells")
    .to_frame()
)

overlap_distribution

count    26612.000000
mean        97.295287
std         13.533215
min          0.001282
1%          14.851600
5%          98.897234
10%        100.000000
25%        100.000000
50%        100.000000
75%        100.000000
90%        100.000000
95%        100.000000
99%        100.000000
max        100.000000
Name: canada_area_pct, dtype: float64


,cells
canada_area_pct,
"[0.0, 1.0)",44
"[1.0, 10.0)",159
"[10.0, 25.0)",178
"[25.0, 50.0)",369
"[50.0, 75.0)",305
"[75.0, 90.0)",162
"[90.0, 99.0)",115
"[99.0, 100.0)",25280


In [30]:
# ---------------------------------------------------------------------------
# Attach Canadian province assignment
# ---------------------------------------------------------------------------

saline_canada = (
    saline_canada
    .join(province_assignments)
)

saline_canada["intersects_canada"] = True

print(f"Canadian-intersecting cells: {len(saline_canada):,}")

saline_canada[
    [
        "COL_ROW",
        "RESOURCE_NAME",
        "province",
        "province_name",
        "province_assignment_method",
        "canada_area_pct",
    ]
].head()

Canadian-intersecting cells: 26,612


,COL_ROW,RESOURCE_NAME,province,province_name,province_assignment_method,canada_area_pct
0,314 - 367,Basal Cambrian,AB,Alberta,single_intersection,100.0
1,314 - 366,Basal Cambrian,AB,Alberta,single_intersection,100.0
2,314 - 365,Basal Cambrian,AB,Alberta,single_intersection,100.0
3,314 - 364,Basal Cambrian,AB,Alberta,single_intersection,100.0
4,315 - 367,Basal Cambrian,AB,Alberta,single_intersection,100.0


In [31]:
# ---------------------------------------------------------------------------
# Classify Canadian spatial coverage
# ---------------------------------------------------------------------------

saline_canada["canada_overlap_class"] = pd.cut(
    saline_canada["canada_area_pct"],
    bins=[
        0,
        1,
        50,
        99,
        100.0001,
    ],
    labels=[
        "trace_intersection",
        "minority_canada",
        "majority_canada",
        "effectively_canada",
    ],
    right=False,
)

saline_canada[
    "canada_overlap_class"
].value_counts().sort_index()

canada_overlap_class
trace_intersection       44
minority_canada         706
majority_canada         582
effectively_canada    25280
Name: count, dtype: int64

In [32]:
# ---------------------------------------------------------------------------
# Summarize Canadian saline storage by province
# ---------------------------------------------------------------------------

province_summary = (
    saline_canada
    .groupby(
        ["province", "province_name"],
        dropna=False,
    )
    .agg(
        cells=("COL_ROW", "count"),
        assessed_cells=("ASSESSED", lambda s: s.eq(1).sum()),
        duplicate_cells=("DUPLICATE", lambda s: s.eq(1).sum()),
        overlap_cells=("OVERLAP", lambda s: s.eq(1).sum()),
        resource_count=("RESOURCE_NAME", "nunique"),
        p50_total_t=("VOL_MED", "sum"),
        mean_canada_area_pct=("canada_area_pct", "mean"),
    )
    .reset_index()
)

province_summary["assessed_pct"] = (
    province_summary["assessed_cells"]
    / province_summary["cells"]
    * 100
).round(2)

province_summary["duplicate_pct"] = (
    province_summary["duplicate_cells"]
    / province_summary["cells"]
    * 100
).round(2)

province_summary["overlap_pct"] = (
    province_summary["overlap_cells"]
    / province_summary["cells"]
    * 100
).round(2)

province_summary["p50_total_Gt"] = (
    province_summary["p50_total_t"]
    / 1e9
)

province_summary.sort_values(
    "p50_total_Gt",
    ascending=False,
)

,province,province_name,cells,assessed_cells,duplicate_cells,overlap_cells,resource_count,p50_total_t,mean_canada_area_pct,assessed_pct,duplicate_pct,overlap_pct,p50_total_Gt
5,SK,Saskatchewan,6943,6943,12,3989,27,2.895652e+11,95.224973,100.00,0.17,57.45,289.565204
0,AB,Alberta,12435,12351,6,4003,19,7.788620e+10,98.642339,99.32,0.05,32.19,77.886197
2,MB,Manitoba,1882,1882,0,1880,3,1.361305e+10,98.571680,100.00,0.00,99.89,13.613051
1,BC,British Columbia,4974,401,65,1324,47,1.975024e+09,97.090153,8.06,1.31,26.62,1.975024
3,NT,Northwest Territories,196,182,0,10,4,1.933599e+08,100.000000,92.86,0.00,5.10,0.193360
4,ON,Ontario,59,59,0,13,9,2.409126e+07,40.588843,100.00,0.00,22.03,0.024091
6,YT,Yukon,123,0,0,0,5,0.000000e+00,89.631079,0.00,0.00,0.00,0.000000


In [34]:
# ---------------------------------------------------------------------------
# Inventory saline resources by province
# Preserve null capacity when no estimates are available
# ---------------------------------------------------------------------------

resource_by_province = (
    saline_canada
    .groupby(
        [
            "province",
            "province_name",
            "RESOURCE_NAME",
        ],
        dropna=False,
    )
    .agg(
        cells=("COL_ROW", "count"),
        assessed_cells=("ASSESSED", lambda s: s.eq(1).sum()),
        p50_estimate_cells=("VOL_MED", "count"),
        p50_total_t=(
            "VOL_MED",
            lambda s: s.sum(min_count=1),
        ),
    )
    .reset_index()
)

resource_by_province["assessed_pct"] = (
    resource_by_province["assessed_cells"]
    / resource_by_province["cells"]
    * 100
).round(2)

resource_by_province["p50_total_Gt"] = (
    resource_by_province["p50_total_t"]
    / 1e9
)

resource_by_province.sort_values(
    ["province", "p50_total_Gt"],
    ascending=[True, False],
)

,province,province_name,RESOURCE_NAME,cells,assessed_cells,p50_estimate_cells,p50_total_t,assessed_pct,p50_total_Gt
0,AB,Alberta,Basal Cambrian,2659,2659,2659,3.448826e+10,100.00,34.488259
16,AB,Alberta,Viking,1290,1260,1260,3.280593e+10,97.67,32.805929
11,AB,Alberta,Rundle Group,1275,1275,1275,2.030705e+09,100.00,2.030705
4,AB,Alberta,Elk Point Group,1929,1929,1929,2.026098e+09,100.00,2.026098
17,AB,Alberta,Winterburn Group,1934,1934,1934,1.829963e+09,100.00,1.829963
...,...,...,...,...,...,...,...,...,...
109,YT,Yukon,Kaktovik Basin,4,0,0,NaN,0.00,NaN
110,YT,Yukon,Kandik Basin,104,0,0,NaN,0.00,NaN
111,YT,Yukon,Liard Basin,6,0,0,NaN,0.00,NaN
112,YT,Yukon,McDame Basin,1,0,0,NaN,0.00,NaN


In [35]:
# ---------------------------------------------------------------------------
# Inspect Canadian resource / basin / partnership structure
# ---------------------------------------------------------------------------

structure_summary = pd.DataFrame(
    {
        "non_null": saline_canada[
            [
                "RESOURCE_NAME",
                "BASIN_NAME",
                "PARTNERSHIP",
                "ARRA_PROJECT",
            ]
        ].notna().sum(),
        "unique_values": saline_canada[
            [
                "RESOURCE_NAME",
                "BASIN_NAME",
                "PARTNERSHIP",
                "ARRA_PROJECT",
            ]
        ].nunique(dropna=True),
    }
)

structure_summary["non_null_pct"] = (
    structure_summary["non_null"]
    / len(saline_canada)
    * 100
).round(2)

structure_summary

,non_null,unique_values,non_null_pct
RESOURCE_NAME,26612,81,100.00
BASIN_NAME,700,4,2.63
PARTNERSHIP,26612,4,100.00
ARRA_PROJECT,0,0,0.00


In [36]:
# ---------------------------------------------------------------------------
# Resource / basin / partnership combinations
# ---------------------------------------------------------------------------

resource_structure = (
    saline_canada
    .groupby(
        [
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
        ],
        dropna=False,
    )
    .agg(
        cells=("COL_ROW", "count"),
        assessed_cells=("ASSESSED", lambda s: s.eq(1).sum()),
        p50_total_t=("VOL_MED", lambda s: s.sum(min_count=1)),
    )
    .reset_index()
)

resource_structure["p50_total_Gt"] = (
    resource_structure["p50_total_t"] / 1e9
)

resource_structure.sort_values(
    ["PARTNERSHIP", "BASIN_NAME", "RESOURCE_NAME"],
)

,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,cells,assessed_cells,p50_total_t,p50_total_Gt
0,BSCSP,Central Montana Uplift,Blackleaf,38,38,6.440230e+08,0.644023
1,BSCSP,Central Montana Uplift,Charles,3,3,2.995336e+07,0.029953
2,BSCSP,Central Montana Uplift,Duperow,41,41,4.763393e+08,0.476339
3,BSCSP,Central Montana Uplift,Greenhorn,11,11,3.471911e+07,0.034719
4,BSCSP,Central Montana Uplift,Kootenai,41,41,2.251489e+08,0.225149
...,...,...,...,...,...,...,...
92,WESTCARB,NaN,Tyaughton Trough,147,0,NaN,NaN
93,WESTCARB,NaN,Watson Lake Basin,2,0,NaN,NaN
94,WESTCARB,NaN,Whatcom,6,6,1.498787e+08,0.149879
95,WESTCARB,NaN,White Lake Basin,4,0,NaN,NaN


In [37]:
# ---------------------------------------------------------------------------
# Check whether resource names occur in multiple basins or partnerships
# ---------------------------------------------------------------------------

resource_identity_check = (
    saline_canada
    .groupby("RESOURCE_NAME")
    .agg(
        basin_count=("BASIN_NAME", "nunique"),
        partnership_count=("PARTNERSHIP", "nunique"),
        province_count=("province", "nunique"),
        cells=("COL_ROW", "count"),
    )
    .sort_values(
        ["basin_count", "partnership_count", "province_count"],
        ascending=False,
    )
)

resource_identity_check.head(30)

,basin_count,partnership_count,province_count,cells
RESOURCE_NAME,,,,
Duperow,3,1,2,64
Madison,3,1,2,39
Nisku,3,1,2,98
SourisRiver,3,1,2,60
ThreeForks,3,1,2,62
Mission Canyon,2,2,2,310
Blackleaf,2,1,2,41
Sawtooth,2,1,2,40
Swift,2,1,2,40


In [38]:
# ---------------------------------------------------------------------------
# Check whether NATCARB grid cells contain multiple storage resources
# ---------------------------------------------------------------------------

cell_identity = (
    saline_canada
    .groupby("COL_ROW")
    .agg(
        records=("COL_ROW", "size"),
        resource_count=("RESOURCE_NAME", "nunique"),
        partnership_count=("PARTNERSHIP", "nunique"),
        basin_count=("BASIN_NAME", "nunique"),
        province_count=("province", "nunique"),
    )
)

print(f"Unique COL_ROW values: {len(cell_identity):,}")
print(f"Total records:         {len(saline_canada):,}")

print()
print("Records per grid cell:")
print(
    cell_identity["records"]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("Resources per grid cell:")
print(
    cell_identity["resource_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

Unique COL_ROW values: 14,789
Total records:         26,612

Records per grid cell:
records
1     9487
2     2075
3     1405
4     1090
5      437
6      235
7        2
8        2
9        1
10       6
11       1
12       8
13      14
14       9
15       3
16       5
17       1
18       5
19       2
21       1

Resources per grid cell:
resource_count
1     9487
2     2075
3     1405
4     1091
5      436
6      235
7        4
9        3
10       5
11       6
12      18
13      10
14       4
15       4
16       1
17       5


In [39]:
# ---------------------------------------------------------------------------
# Inspect multi-resource NATCARB grid cells
# ---------------------------------------------------------------------------

multi_resource_cells = (
    cell_identity[
        cell_identity["resource_count"] > 1
    ]
    .index
)

multi_resource_records = (
    saline_canada[
        saline_canada["COL_ROW"].isin(
            multi_resource_cells
        )
    ]
    [
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
            "VOL_MED",
            "ASSESSED",
            "OVERLAP",
            "DUPLICATE",
            "province",
        ]
    ]
    .sort_values(
        ["COL_ROW", "RESOURCE_NAME"]
    )
)

print(
    f"Grid cells with multiple resources: "
    f"{len(multi_resource_cells):,}"
)

multi_resource_records.head(30)

Grid cells with multiple resources: 5,302


,COL_ROW,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,VOL_MED,ASSESSED,OVERLAP,DUPLICATE,province
39895,233 - 359,WESTCARB,NaN,Hecate Basin,NaN,0,0,0,BC
39896,233 - 359,WESTCARB,NaN,Nanaimo Subbasin,NaN,0,0,0,BC
39893,233 - 360,WESTCARB,NaN,Hecate Basin,NaN,0,0,0,BC
39894,233 - 360,WESTCARB,NaN,Nanaimo Subbasin,NaN,0,0,0,BC
40017,234 - 348,WESTCARB,NaN,Hecate Basin,NaN,0,0,0,BC
40018,234 - 348,WESTCARB,NaN,Queen Charlotte Basin,NaN,0,0,0,BC
40015,234 - 349,WESTCARB,NaN,Hecate Basin,NaN,0,0,0,BC
40016,234 - 349,WESTCARB,NaN,Queen Charlotte Basin,NaN,0,0,0,BC
40013,234 - 350,WESTCARB,NaN,Hecate Basin,NaN,0,0,0,BC
40014,234 - 350,WESTCARB,NaN,Queen Charlotte Basin,NaN,0,0,0,BC


In [40]:
# ---------------------------------------------------------------------------
# Test candidate source-record identifiers
# ---------------------------------------------------------------------------

candidate_key_cols = [
    "COL_ROW",
    "PARTNERSHIP",
    "RESOURCE_NAME",
    "BASIN_NAME",
]

candidate_key = (
    saline_canada[candidate_key_cols]
    .fillna("<NULL>")
)

duplicate_candidate_key = (
    candidate_key
    .duplicated(keep=False)
)

print(
    f"Records: {len(saline_canada):,}"
)

print(
    f"Unique COL_ROW values: "
    f"{saline_canada['COL_ROW'].nunique():,}"
)

print(
    f"Unique composite keys: "
    f"{len(candidate_key.drop_duplicates()):,}"
)

print(
    f"Records involved in duplicate composite keys: "
    f"{duplicate_candidate_key.sum():,}"
)

Records: 26,612
Unique COL_ROW values: 14,789
Unique composite keys: 26,580
Records involved in duplicate composite keys: 64


In [41]:
# ---------------------------------------------------------------------------
# Inspect duplicate candidate source-record keys
# ---------------------------------------------------------------------------

duplicate_key_records = (
    saline_canada.loc[
        duplicate_candidate_key,
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
            "VOL_LOW",
            "VOL_MED",
            "VOL_HIGH",
            "ASSESSED",
            "OVERLAP",
            "DUPLICATE",
            "RSC_AREA_CELL",
        ],
    ]
    .sort_values(
        [
            "COL_ROW",
            "PARTNERSHIP",
            "RESOURCE_NAME",
        ]
    )
)

duplicate_key_records.head(50)

,COL_ROW,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,VOL_LOW,VOL_MED,VOL_HIGH,ASSESSED,OVERLAP,DUPLICATE,RSC_AREA_CELL
143164,348 - 438,BSCSP,Central Montana Uplift,Nisku,4.039019e+05,1.388413e+06,3.471032e+06,1,0,0,3.680671e+07
143168,348 - 438,BSCSP,Central Montana Uplift,Nisku,4.039019e+05,1.388413e+06,3.471032e+06,1,0,0,3.680671e+07
143468,349 - 438,BSCSP,Central Montana Uplift,Nisku,6.836560e+05,2.350068e+06,5.875169e+06,1,0,0,6.230011e+07
143472,349 - 438,BSCSP,Central Montana Uplift,Nisku,6.836560e+05,2.350068e+06,5.875169e+06,1,0,0,6.230011e+07
143855,350 - 438,BSCSP,Central Montana Uplift,Nisku,6.291357e+05,2.162654e+06,5.406635e+06,1,0,0,5.733180e+07
143859,350 - 438,BSCSP,Central Montana Uplift,Nisku,6.291357e+05,2.162654e+06,5.406635e+06,1,0,0,5.733180e+07
144337,351 - 438,BSCSP,Central Montana Uplift,Nisku,5.191142e+05,1.784455e+06,4.461137e+06,1,0,0,4.730576e+07
144341,351 - 438,BSCSP,Central Montana Uplift,Nisku,5.191142e+05,1.784455e+06,4.461137e+06,1,0,0,4.730576e+07
144886,352 - 438,BSCSP,Central Montana Uplift,Nisku,3.571125e+05,1.227574e+06,3.068936e+06,1,0,0,3.254290e+07
144890,352 - 438,BSCSP,Central Montana Uplift,Nisku,3.571125e+05,1.227574e+06,3.068936e+06,1,0,0,3.254290e+07


In [42]:
# ---------------------------------------------------------------------------
# Verify geometry consistency within COL_ROW
# ---------------------------------------------------------------------------

geometry_per_cell = (
    saline_canada
    .assign(
        geometry_wkb=saline_canada.geometry.to_wkb()
    )
    .groupby("COL_ROW")
    ["geometry_wkb"]
    .nunique()
)

print(
    "Cells with multiple distinct geometries: "
    f"{(geometry_per_cell > 1).sum():,}"
)

geometry_per_cell.value_counts().sort_index()

Cells with multiple distinct geometries: 4,022


geometry_wkb
1    10767
2     4022
Name: count, dtype: int64

In [43]:
# ---------------------------------------------------------------------------
# Diagnose duplicate composite source-record keys
# ---------------------------------------------------------------------------

duplicate_key_diagnostics = (
    saline_canada.loc[
        duplicate_candidate_key
    ]
    .assign(
        geometry_wkb=lambda df: df.geometry.to_wkb(),
        geometry_area_m2=lambda df: df.geometry.area,
    )
    .groupby(
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
        ],
        dropna=False,
    )
    .agg(
        records=("COL_ROW", "size"),
        distinct_geometries=("geometry_wkb", "nunique"),
        total_geometry_area_m2=("geometry_area_m2", "sum"),
        min_geometry_area_m2=("geometry_area_m2", "min"),
        max_geometry_area_m2=("geometry_area_m2", "max"),
        rsc_area_values=("RSC_AREA_CELL", "nunique"),
        vol_med_values=("VOL_MED", "nunique"),
    )
    .reset_index()
)

duplicate_key_diagnostics.sort_values(
    ["records", "distinct_geometries"],
    ascending=False,
)

,COL_ROW,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,records,distinct_geometries,total_geometry_area_m2,min_geometry_area_m2,max_geometry_area_m2,rsc_area_values,vol_med_values
0,348 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
1,349 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
2,350 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
3,351 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
4,352 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
5,353 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
6,354 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
7,354 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
8,355 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2
9,356 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,200000000.0,100000000.0,100000000.0,1,2


In [44]:
# ---------------------------------------------------------------------------
# Compare source geometry area with RSC_AREA_CELL
# ---------------------------------------------------------------------------

area_check = saline_canada.assign(
    geometry_area_m2=saline_canada.geometry.area
)

area_check["geometry_minus_shape_area"] = (
    area_check["geometry_area_m2"]
    - area_check["Shape_Area"]
)

area_check["geometry_minus_rsc_area"] = (
    area_check["geometry_area_m2"]
    - area_check["RSC_AREA_CELL"]
)

print("Geometry vs Shape_Area")
print(
    area_check["geometry_minus_shape_area"]
    .abs()
    .describe()
)

print("\nGeometry vs RSC_AREA_CELL")
print(
    area_check["geometry_minus_rsc_area"]
    .abs()
    .describe()
)

Geometry vs Shape_Area
count    26612.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: geometry_minus_shape_area, dtype: float64

Geometry vs RSC_AREA_CELL
count    2.658800e+04
mean     1.167161e+07
std      2.657693e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+08
Name: geometry_minus_rsc_area, dtype: float64


In [46]:
# ---------------------------------------------------------------------------
# Inspect apparent duplicate source records at full numeric precision
# ---------------------------------------------------------------------------

duplicate_detail = (
    saline_canada.loc[
        duplicate_candidate_key,
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
            "RSC_AREA_CELL",
            "VOL_LOW",
            "VOL_MED",
            "VOL_HIGH",
            "DEPTH_FT",
            "THICKNESS_FT",
            "POROSITY_PCT",
            "PERMEABILITY_mD",
            "ASSESSED",
            "OVERLAP",
            "DUPLICATE",
            "geometry",
        ],
    ]
    .copy()
)

duplicate_detail["geometry_wkb"] = (
    duplicate_detail.geometry.to_wkb()
)

numeric_fields = [
    "RSC_AREA_CELL",
    "VOL_LOW",
    "VOL_MED",
    "VOL_HIGH",
    "DEPTH_FT",
    "THICKNESS_FT",
    "POROSITY_PCT",
    "PERMEABILITY_mD",
]

duplicate_variation = (
    duplicate_detail
    .groupby(
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
        ],
        dropna=False,
    )
    .agg(
        records=("COL_ROW", "size"),
        distinct_geometries=("geometry_wkb", "nunique"),
        **{
            f"{field}_nunique": (
                field,
                lambda s: s.nunique(dropna=False),
            )
            for field in numeric_fields
        },
    )
    .reset_index()
)

duplicate_variation

,COL_ROW,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,records,distinct_geometries,RSC_AREA_CELL_nunique,VOL_LOW_nunique,VOL_MED_nunique,VOL_HIGH_nunique,DEPTH_FT_nunique,THICKNESS_FT_nunique,POROSITY_PCT_nunique,PERMEABILITY_mD_nunique
0,348 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
1,349 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
2,350 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
3,351 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
4,352 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
5,353 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
6,354 - 438,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
7,354 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
8,355 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1
9,356 - 439,BSCSP,Central Montana Uplift,Nisku,2,1,1,2,2,2,1,1,1,1


In [47]:
# ---------------------------------------------------------------------------
# Quantify hidden VOL_MED differences
# ---------------------------------------------------------------------------

vol_med_difference = (
    duplicate_detail
    .groupby(
        [
            "COL_ROW",
            "PARTNERSHIP",
            "BASIN_NAME",
            "RESOURCE_NAME",
        ],
        dropna=False,
    )["VOL_MED"]
    .agg(["min", "max"])
)

vol_med_difference["absolute_difference"] = (
    vol_med_difference["max"]
    - vol_med_difference["min"]
)

vol_med_difference["relative_difference_pct"] = (
    vol_med_difference["absolute_difference"]
    / vol_med_difference["max"]
    * 100
)

vol_med_difference.sort_values(
    "relative_difference_pct",
    ascending=False,
).head(40)

,,,,min,max,absolute_difference,relative_difference_pct
COL_ROW,PARTNERSHIP,BASIN_NAME,RESOURCE_NAME,,,,
388 - 443,BSCSP,Central Montana Uplift,Nisku,7.888550e+04,1.103031e+07,1.095142e+07,9.928483e+01
387 - 443,BSCSP,Central Montana Uplift,Nisku,1.080555e+05,2.750300e+06,2.642244e+06,9.607114e+01
354 - 439,BSCSP,Central Montana Uplift,Nisku,2.849978e+06,2.849978e+06,2.345536e-06,8.230013e-11
372 - 441,BSCSP,Central Montana Uplift,Nisku,1.765359e+06,1.765359e+06,1.452863e-06,8.229844e-11
377 - 442,BSCSP,Central Montana Uplift,Nisku,2.949106e+06,2.949106e+06,2.427027e-06,8.229704e-11
350 - 438,BSCSP,Central Montana Uplift,Nisku,2.162654e+06,2.162654e+06,1.779757e-06,8.229506e-11
379 - 442,BSCSP,Central Montana Uplift,Nisku,1.925575e+06,1.925575e+06,1.584645e-06,8.229466e-11
380 - 442,BSCSP,Central Montana Uplift,Nisku,1.416339e+06,1.416339e+06,1.165550e-06,8.229318e-11
355 - 439,BSCSP,Central Montana Uplift,Nisku,2.308725e+06,2.308725e+06,1.899898e-06,8.229207e-11


In [48]:
# ---------------------------------------------------------------------------
# Characterize geometry area versus reported resource area
# ---------------------------------------------------------------------------

area_check["resource_area_fraction_of_geometry"] = (
    area_check["RSC_AREA_CELL"]
    / area_check["geometry_area_m2"]
)

area_relation_summary = pd.Series(
    {
        "records": len(area_check),
        "exact_area_match": (
            area_check["geometry_area_m2"]
            == area_check["RSC_AREA_CELL"]
        ).sum(),
        "geometry_larger_than_resource": (
            area_check["geometry_area_m2"]
            > area_check["RSC_AREA_CELL"]
        ).sum(),
        "geometry_smaller_than_resource": (
            area_check["geometry_area_m2"]
            < area_check["RSC_AREA_CELL"]
        ).sum(),
    },
    name="records",
).to_frame()

area_relation_summary["share_pct"] = (
    area_relation_summary["records"]
    / len(area_check)
    * 100
).round(2)

area_relation_summary

,records,share_pct
records,26612,100.00
exact_area_match,20690,77.75
geometry_larger_than_resource,5898,22.16
geometry_smaller_than_resource,0,0.00


In [49]:
# ---------------------------------------------------------------------------
# Reload saline layer with source feature ID
# ---------------------------------------------------------------------------

saline_10k_gdf_fid = pyogrio.read_dataframe(
    GDB_PATH,
    layer="NATCARB_Saline_10K_v1502",
    fid_as_index=True,
)

saline_10k_gdf_fid.index.name = "source_fid"

print(saline_10k_gdf_fid.index[:10])
print(f"Unique FIDs: {saline_10k_gdf_fid.index.nunique():,}")
print(f"Records:     {len(saline_10k_gdf_fid):,}")

Index([810001, 810002, 810003, 810004, 810005, 810006, 810007, 810008, 810009,
       810010],
      dtype='int64', name='source_fid')
Unique FIDs: 186,675
Records:     186,675


In [50]:
# ---------------------------------------------------------------------------
# Inspect genuinely distinct records sharing the same descriptive key
# ---------------------------------------------------------------------------

problem_cells = [
    "387 - 443",
    "388 - 443",
]

problem_records = (
    saline_canada[
        saline_canada["COL_ROW"].isin(problem_cells)
        & saline_canada["PARTNERSHIP"].eq("BSCSP")
        & saline_canada["RESOURCE_NAME"].eq("Nisku")
    ]
    .sort_values(["COL_ROW", "VOL_MED"])
)

pd.set_option("display.max_columns", None)

problem_records

,COL_ROW,PARTNERSHIP,ARRA_PROJECT,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,VOL_LOW,VOL_MED,VOL_HIGH,DEPTH_FT,THICKNESS_FT,SALINITY_TDS,PRESSURE_PSI,TEMPERATURE_F,POROSITY_PCT,PERMEABILITY_mD,ASSESSED,CYCLE_OF_LAST_UPDATE,OVERLAP,DUPLICATE,MED_CALCED,Shape_Length,Shape_Area,geometry,canada_intersection_area_m2,cell_area_m2,canada_area_fraction,canada_area_pct,province,province_name,province_assignment_method,intersects_canada,canada_overlap_class
168987,387 - 443,BSCSP,NaN,Nisku,Central Montana Uplift,6.595128e+05,3.143434e+04,1.080555e+05,2.701388e+05,4084.0,73.0,NaN,NaN,NaN,NaN,NaN,1,"Atlas V, v2",0,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-541251.74 458935.68, -541251....",1.422511e+05,100000000.0,0.001423,0.142251,SK,Saskatchewan,single_intersection,True,trace_intersection
168984,387 - 443,BSCSP,NaN,Nisku,Central Montana Uplift,7.291024e+07,8.000873e+05,2.750300e+06,6.875750e+06,4084.0,73.0,NaN,NaN,NaN,NaN,NaN,1,"Atlas V, v2",0,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-541251.74 458935.68, -541251....",1.422511e+05,100000000.0,0.001423,0.142251,SK,Saskatchewan,single_intersection,True,trace_intersection
169670,388 - 443,BSCSP,NaN,Nisku,Central Montana Uplift,2.091249e+06,2.294851e+04,7.888550e+04,1.972137e+05,4084.0,73.0,NaN,NaN,NaN,NaN,NaN,1,"Atlas V, v2",0,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-531251.74 458935.68, -531251....",6.137936e+06,100000000.0,0.061379,6.137936,SK,Saskatchewan,single_intersection,True,minority_canada
169672,388 - 443,BSCSP,NaN,Nisku,Central Montana Uplift,6.732305e+07,3.208816e+06,1.103031e+07,2.757577e+07,4084.0,73.0,NaN,NaN,NaN,NaN,NaN,1,"Atlas V, v2",0,0,0,40000.0,100000000.0,"MULTIPOLYGON (((-531251.74 458935.68, -531251....",6.137936e+06,100000000.0,0.061379,6.137936,SK,Saskatchewan,single_intersection,True,minority_canada


In [51]:
# ---------------------------------------------------------------------------
# Attach native FileGDB feature ID to existing Canadian subset
# and standardize geometry-area naming
# ---------------------------------------------------------------------------

# Build a lookup from the original GeoDataFrame index
# to the native FileGDB feature ID.
source_fid_lookup = pd.Series(
    saline_10k_gdf_fid.index.to_numpy(),
    index=saline_10k_gdf.index,
    name="source_fid",
)

# Validate that the standard read and fid_as_index read
# preserve the same source feature ordering.
assert (
    saline_10k_gdf["COL_ROW"].to_numpy()
    == saline_10k_gdf_fid["COL_ROW"].to_numpy()
).all()

# Work on a copy of the Canadian subset.
saline_canada = saline_canada.copy()

# Attach the native FileGDB feature ID.
saline_canada["source_fid"] = (
    saline_canada.index
    .map(source_fid_lookup)
    .astype("int64")
)

# Validate source-record identity.
assert saline_canada["source_fid"].notna().all()
assert saline_canada["source_fid"].is_unique

# Rename this field to reflect what it actually represents:
# the area of the supplied NATCARB feature geometry,
# not necessarily the geological resource area.
saline_canada = saline_canada.rename(
    columns={
        "cell_area_m2": "geometry_area_m2",
    }
)

# ---------------------------------------------------------------------------
# QA summary
# ---------------------------------------------------------------------------

print(
    f"Canadian source records: "
    f"{len(saline_canada):,}"
)

print(
    f"Unique source FIDs:       "
    f"{saline_canada['source_fid'].nunique():,}"
)

print(
    f"Unique COL_ROW values:    "
    f"{saline_canada['COL_ROW'].nunique():,}"
)

print()

saline_canada[
    [
        "source_fid",
        "COL_ROW",
        "PARTNERSHIP",
        "RESOURCE_NAME",
        "BASIN_NAME",
        "RSC_AREA_CELL",
        "geometry_area_m2",
        "province",
        "province_assignment_method",
        "canada_area_pct",
        "canada_overlap_class",
    ]
].head()

Canadian source records: 26,612
Unique source FIDs:       26,612
Unique COL_ROW values:    14,789



,source_fid,COL_ROW,PARTNERSHIP,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,geometry_area_m2,province,province_assignment_method,canada_area_pct,canada_overlap_class
0,810001,314 - 367,PCOR,Basal Cambrian,NaN,11000000.0,100000000.0,AB,single_intersection,100.0,effectively_canada
1,810002,314 - 366,PCOR,Basal Cambrian,NaN,83000000.0,100000000.0,AB,single_intersection,100.0,effectively_canada
2,810003,314 - 365,PCOR,Basal Cambrian,NaN,93000000.0,100000000.0,AB,single_intersection,100.0,effectively_canada
3,810004,314 - 364,PCOR,Basal Cambrian,NaN,11000000.0,100000000.0,AB,single_intersection,100.0,effectively_canada
4,810005,315 - 367,PCOR,Basal Cambrian,NaN,79000000.0,100000000.0,AB,single_intersection,100.0,effectively_canada


In [52]:
# ---------------------------------------------------------------------------
# Summarize assessed storage capacity by Canadian overlap class
# ---------------------------------------------------------------------------

overlap_capacity_summary = (
    saline_canada
    .groupby(
        "canada_overlap_class",
        observed=False,
    )
    .agg(
        records=("source_fid", "count"),
        unique_cells=("COL_ROW", "nunique"),
        assessed_records=("ASSESSED", lambda s: s.eq(1).sum()),
        p50_estimate_records=("VOL_MED", "count"),
        duplicate_records=("DUPLICATE", lambda s: s.eq(1).sum()),
        raw_p50_total_t=(
            "VOL_MED",
            lambda s: s.sum(min_count=1),
        ),
    )
    .reset_index()
)

overlap_capacity_summary["raw_p50_total_Gt"] = (
    overlap_capacity_summary["raw_p50_total_t"]
    / 1e9
)

overlap_capacity_summary["record_share_pct"] = (
    overlap_capacity_summary["records"]
    / len(saline_canada)
    * 100
).round(2)

overlap_capacity_summary

,canada_overlap_class,records,unique_cells,assessed_records,p50_estimate_records,duplicate_records,raw_p50_total_t,raw_p50_total_Gt,record_share_pct
0,trace_intersection,44,18,32,32,0,2.808260e+08,0.280826,0.17
1,minority_canada,706,221,565,565,10,6.133303e+09,6.133303,2.65
2,majority_canada,582,206,416,416,7,2.300973e+09,2.300973,2.19
3,effectively_canada,25280,14344,20805,20805,66,3.745418e+11,374.541826,94.99


In [53]:
# ---------------------------------------------------------------------------
# Non-duplicate P50 totals by Canadian overlap class
# ---------------------------------------------------------------------------

nonduplicate_overlap_capacity = (
    saline_canada[
        saline_canada["DUPLICATE"].eq(0)
    ]
    .groupby(
        "canada_overlap_class",
        observed=False,
    )["VOL_MED"]
    .apply(
        lambda s: s.sum(min_count=1)
    )
    .rename("nonduplicate_p50_total_t")
    .reset_index()
)

overlap_capacity_summary = (
    overlap_capacity_summary
    .merge(
        nonduplicate_overlap_capacity,
        on="canada_overlap_class",
        how="left",
    )
)

overlap_capacity_summary[
    "nonduplicate_p50_total_Gt"
] = (
    overlap_capacity_summary[
        "nonduplicate_p50_total_t"
    ]
    / 1e9
)

overlap_capacity_summary

,canada_overlap_class,records,unique_cells,assessed_records,p50_estimate_records,duplicate_records,raw_p50_total_t,raw_p50_total_Gt,record_share_pct,nonduplicate_p50_total_t,nonduplicate_p50_total_Gt
0,trace_intersection,44,18,32,32,0,2.808260e+08,0.280826,0.17,2.808260e+08,0.280826
1,minority_canada,706,221,565,565,10,6.133303e+09,6.133303,2.65,5.393342e+09,5.393342
2,majority_canada,582,206,416,416,7,2.300973e+09,2.300973,2.19,1.860088e+09,1.860088
3,effectively_canada,25280,14344,20805,20805,66,3.745418e+11,374.541826,94.99,3.745418e+11,374.541826


In [54]:
# ---------------------------------------------------------------------------
# Quantify P50 contribution by Canadian overlap class
# ---------------------------------------------------------------------------

total_nonduplicate_p50_t = (
    overlap_capacity_summary[
        "nonduplicate_p50_total_t"
    ]
    .sum(min_count=1)
)

overlap_capacity_summary[
    "nonduplicate_p50_share_pct"
] = (
    overlap_capacity_summary[
        "nonduplicate_p50_total_t"
    ]
    / total_nonduplicate_p50_t
    * 100
).round(3)

print(
    f"Total non-duplicate P50: "
    f"{total_nonduplicate_p50_t / 1e9:.3f} Gt"
)

display(
    overlap_capacity_summary[
        [
            "canada_overlap_class",
            "records",
            "assessed_records",
            "duplicate_records",
            "nonduplicate_p50_total_Gt",
            "nonduplicate_p50_share_pct",
        ]
    ]
)

# ---------------------------------------------------------------------------
# Inspect duplicate records by overlap class
# ---------------------------------------------------------------------------

duplicate_capacity_check = (
    saline_canada[
        saline_canada["DUPLICATE"].eq(1)
    ]
    .groupby(
        "canada_overlap_class",
        observed=False,
    )
    .agg(
        duplicate_records=("source_fid", "count"),
        p50_nonnull_records=("VOL_MED", "count"),
        duplicate_p50_total_t=(
            "VOL_MED",
            lambda s: s.sum(min_count=1),
        ),
    )
    .reset_index()
)

duplicate_capacity_check[
    "duplicate_p50_total_Gt"
] = (
    duplicate_capacity_check[
        "duplicate_p50_total_t"
    ]
    / 1e9
)

duplicate_capacity_check

Total non-duplicate P50: 382.076 Gt


,canada_overlap_class,records,assessed_records,duplicate_records,nonduplicate_p50_total_Gt,nonduplicate_p50_share_pct
0,trace_intersection,44,32,0,0.280826,0.074
1,minority_canada,706,565,10,5.393342,1.412
2,majority_canada,582,416,7,1.860088,0.487
3,effectively_canada,25280,20805,66,374.541826,98.028


,canada_overlap_class,duplicate_records,p50_nonnull_records,duplicate_p50_total_t,duplicate_p50_total_Gt
0,trace_intersection,0,0,NaN,NaN
1,minority_canada,10,8,7.399611e+08,0.739961
2,majority_canada,7,4,4.408856e+08,0.440886
3,effectively_canada,66,0,NaN,NaN


In [55]:
# ---------------------------------------------------------------------------
# Summarize Canadian overlap classes by province
# ---------------------------------------------------------------------------

province_overlap_summary = (
    saline_canada
    .groupby(
        [
            "province",
            "province_name",
            "canada_overlap_class",
        ],
        observed=False,
        dropna=False,
    )
    .agg(
        records=("source_fid", "count"),
        unique_cells=("COL_ROW", "nunique"),
        assessed_records=("ASSESSED", lambda s: s.eq(1).sum()),
        duplicate_records=("DUPLICATE", lambda s: s.eq(1).sum()),
        raw_p50_total_t=(
            "VOL_MED",
            lambda s: s.sum(min_count=1),
        ),
    )
    .reset_index()
)

province_overlap_summary["raw_p50_total_Gt"] = (
    province_overlap_summary["raw_p50_total_t"]
    / 1e9
)

# ---------------------------------------------------------------------------
# Add non-duplicate P50 totals
# ---------------------------------------------------------------------------

province_overlap_nonduplicate = (
    saline_canada[
        saline_canada["DUPLICATE"].eq(0)
    ]
    .groupby(
        [
            "province",
            "province_name",
            "canada_overlap_class",
        ],
        observed=False,
        dropna=False,
    )["VOL_MED"]
    .apply(
        lambda s: s.sum(min_count=1)
    )
    .rename("nonduplicate_p50_total_t")
    .reset_index()
)

province_overlap_summary = (
    province_overlap_summary
    .merge(
        province_overlap_nonduplicate,
        on=[
            "province",
            "province_name",
            "canada_overlap_class",
        ],
        how="left",
    )
)

province_overlap_summary[
    "nonduplicate_p50_total_Gt"
] = (
    province_overlap_summary[
        "nonduplicate_p50_total_t"
    ]
    / 1e9
)

# ---------------------------------------------------------------------------
# Calculate each overlap class's share of provincial P50
# ---------------------------------------------------------------------------

province_totals = (
    province_overlap_summary
    .groupby(
        ["province", "province_name"],
        dropna=False,
    )["nonduplicate_p50_total_t"]
    .sum(min_count=1)
    .rename("province_p50_total_t")
    .reset_index()
)

province_overlap_summary = (
    province_overlap_summary
    .merge(
        province_totals,
        on=["province", "province_name"],
        how="left",
    )
)

province_overlap_summary[
    "province_p50_share_pct"
] = (
    province_overlap_summary[
        "nonduplicate_p50_total_t"
    ]
    / province_overlap_summary[
        "province_p50_total_t"
    ]
    * 100
).round(3)

province_overlap_summary.sort_values(
    [
        "province",
        "canada_overlap_class",
    ]
)

,province,province_name,canada_overlap_class,records,unique_cells,assessed_records,duplicate_records,raw_p50_total_t,raw_p50_total_Gt,nonduplicate_p50_total_t,nonduplicate_p50_total_Gt,province_p50_total_t,province_p50_share_pct
0,AB,Alberta,trace_intersection,0,0,0,0,NaN,NaN,NaN,NaN,7.788620e+10,NaN
1,AB,Alberta,minority_canada,165,12,165,0,1.283238e+09,1.283238,1.283238e+09,1.283238,7.788620e+10,1.648
2,AB,Alberta,majority_canada,143,13,143,0,4.725047e+08,0.472505,4.725047e+08,0.472505,7.788620e+10,0.607
3,AB,Alberta,effectively_canada,12127,4479,12043,6,7.613045e+10,76.130454,7.613045e+10,76.130454,7.788620e+10,97.746
4,BC,British Columbia,trace_intersection,11,11,0,0,NaN,NaN,NaN,NaN,1.975024e+09,NaN
5,BC,British Columbia,minority_canada,132,119,3,2,1.247944e+08,0.124794,1.247944e+08,0.124794,1.975024e+09,6.319
6,BC,British Columbia,majority_canada,158,142,2,3,2.508041e+07,0.025080,2.508041e+07,0.025080,1.975024e+09,1.270
7,BC,British Columbia,effectively_canada,4673,4118,396,60,1.825150e+09,1.825150,1.825150e+09,1.825150,1.975024e+09,92.411
8,MB,Manitoba,trace_intersection,1,1,1,0,0.000000e+00,0.000000,0.000000e+00,0.000000,1.361305e+10,0.000
9,MB,Manitoba,minority_canada,38,32,38,0,4.935051e+08,0.493505,4.935051e+08,0.493505,1.361305e+10,3.625


In [56]:
# ---------------------------------------------------------------------------
# Provincial P50 by Canadian overlap class
# ---------------------------------------------------------------------------

province_overlap_pivot = (
    province_overlap_summary
    .pivot(
        index=["province", "province_name"],
        columns="canada_overlap_class",
        values="nonduplicate_p50_total_Gt",
    )
    .fillna(0)
)

province_overlap_pivot

,canada_overlap_class,trace_intersection,minority_canada,majority_canada,effectively_canada
province,province_name,,,,
AB,Alberta,0.000000,1.283238,0.472505,76.130454
BC,British Columbia,0.000000,0.124794,0.025080,1.825150
MB,Manitoba,0.000000,0.493505,0.000000,13.119546
NT,Northwest Territories,0.000000,0.000000,0.000000,0.193360
ON,Ontario,0.011706,0.009898,0.002488,0.000000
SK,Saskatchewan,0.269120,3.481906,1.360015,283.273316
YT,Yukon,0.000000,0.000000,0.000000,0.000000


In [60]:
# ---------------------------------------------------------------------------
# Inspect geological resources in Canadian boundary-overlap records
# Only retain category combinations that actually occur
# ---------------------------------------------------------------------------

boundary_resource_summary = (
    saline_canada[
        saline_canada["canada_overlap_class"]
        != "effectively_canada"
    ]
    .groupby(
        [
            "province",
            "province_name",
            "canada_overlap_class",
            "PARTNERSHIP",
            "RESOURCE_NAME",
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        records=("source_fid", "count"),
        unique_cells=("COL_ROW", "nunique"),
        assessed_records=("ASSESSED", lambda s: s.eq(1).sum()),
        duplicate_records=("DUPLICATE", lambda s: s.eq(1).sum()),
        p50_total_t=(
            "VOL_MED",
            lambda s: s.sum(min_count=1),
        ),
        mean_canada_area_pct=("canada_area_pct", "mean"),
        min_canada_area_pct=("canada_area_pct", "min"),
        max_canada_area_pct=("canada_area_pct", "max"),
    )
    .reset_index()
)

boundary_resource_summary["p50_total_Gt"] = (
    boundary_resource_summary["p50_total_t"] / 1e9
)

boundary_resource_summary = (
    boundary_resource_summary
    .sort_values(
        [
            "province",
            "p50_total_Gt",
            "records",
        ],
        ascending=[
            True,
            False,
            False,
        ],
        na_position="last",
    )
)

boundary_resource_summary

,province,province_name,canada_overlap_class,PARTNERSHIP,RESOURCE_NAME,records,unique_cells,assessed_records,duplicate_records,p50_total_t,mean_canada_area_pct,min_canada_area_pct,max_canada_area_pct,p50_total_Gt
3,AB,Alberta,minority_canada,BSCSP,Madison,15,12,15,0,2.987008e+08,19.185746,1.538058,44.796032,0.298701
0,AB,Alberta,minority_canada,BSCSP,Blackleaf,14,11,14,0,2.891707e+08,20.135702,1.538058,44.796032,0.289171
1,AB,Alberta,minority_canada,BSCSP,Duperow,14,11,14,0,1.924393e+08,20.135702,1.538058,44.796032,0.192439
18,AB,Alberta,majority_canada,BSCSP,Madison,11,11,11,0,9.841062e+07,74.005533,52.439874,98.352025,0.098411
15,AB,Alberta,majority_canada,BSCSP,Blackleaf,11,11,11,0,9.500778e+07,74.005533,52.439874,98.352025,0.095008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,YT,Yukon,minority_canada,WESTCARB,Kandik Basin,9,9,0,0,NaN,16.988071,1.923907,40.210665,NaN
164,YT,Yukon,majority_canada,WESTCARB,Kandik Basin,9,9,0,0,NaN,78.953392,53.064802,96.482208,NaN
161,YT,Yukon,minority_canada,WESTCARB,Kaktovik Basin,3,3,0,0,NaN,30.480829,20.597143,39.585911,NaN
160,YT,Yukon,trace_intersection,WESTCARB,Kandik Basin,1,1,0,0,NaN,0.069827,0.069827,0.069827,NaN


In [61]:
# ---------------------------------------------------------------------------
# Inspect Ontario NATCARB saline records
# ---------------------------------------------------------------------------

ontario_records = (
    saline_canada[
        saline_canada["province"].eq("ON")
    ]
    [
        [
            "source_fid",
            "COL_ROW",
            "PARTNERSHIP",
            "RESOURCE_NAME",
            "BASIN_NAME",
            "RSC_AREA_CELL",
            "VOL_MED",
            "ASSESSED",
            "DUPLICATE",
            "canada_area_pct",
            "canada_overlap_class",
            "geometry",
        ]
    ]
    .sort_values(
        [
            "canada_area_pct",
            "RESOURCE_NAME",
        ]
    )
)

print(f"Ontario source records: {len(ontario_records):,}")
print(
    f"Ontario unique cells: "
    f"{ontario_records['COL_ROW'].nunique():,}"
)

ontario_records

Ontario source records: 59
Ontario unique cells: 21


,source_fid,COL_ROW,PARTNERSHIP,RESOURCE_NAME,BASIN_NAME,RSC_AREA_CELL,VOL_MED,ASSESSED,DUPLICATE,canada_area_pct,canada_overlap_class,geometry
124715,934716,611 - 491,MRCSP,Bass Island dolomite,NaN,0.0,0.0,1,0,0.019828,trace_intersection,"MULTIPOLYGON (((1698748.26 -21064.32, 1698748...."
128015,938016,611 - 491,MRCSP,Lockport Dolomite,NaN,0.0,0.0,1,0,0.019828,trace_intersection,"MULTIPOLYGON (((1698748.26 -21064.32, 1698748...."
138791,948792,611 - 491,MRCSP,Oriskany sandstone,NaN,NaN,0.0,1,0,0.019828,trace_intersection,"MULTIPOLYGON (((1698748.26 -21064.32, 1698748...."
134096,944097,611 - 491,MRCSP,Rose Run Sandstone,NaN,49574750.0,2145410.0,1,0,0.019828,trace_intersection,"MULTIPOLYGON (((1698748.26 -21064.32, 1698748...."
131350,941351,582 - 498,MRCSP,Basal Sands - Mt. Simon,NaN,97477157.0,9560450.0,1,0,0.049327,trace_intersection,"MULTIPOLYGON (((1408748.26 -91064.32, 1408748...."
126181,936182,582 - 498,MRCSP,Dundee Limestone,NaN,NaN,0.0,1,0,0.049327,trace_intersection,"MULTIPOLYGON (((1408748.26 -91064.32, 1408748...."
136011,946012,582 - 498,MRCSP,Sylvania sandstone,NaN,0.0,0.0,1,0,0.049327,trace_intersection,"MULTIPOLYGON (((1408748.26 -91064.32, 1408748...."
131347,941348,582 - 501,MRCSP,Basal Sands - Mt. Simon,NaN,0.0,0.0,1,0,3.449650,minority_canada,"MULTIPOLYGON (((1408748.26 -121064.32, 1408748..."
126178,936179,582 - 501,MRCSP,Dundee Limestone,NaN,NaN,0.0,1,0,3.449650,minority_canada,"MULTIPOLYGON (((1408748.26 -121064.32, 1408748..."
136008,946009,582 - 501,MRCSP,Sylvania sandstone,NaN,0.0,0.0,1,0,3.449650,minority_canada,"MULTIPOLYGON (((1408748.26 -121064.32, 1408748..."


In [62]:
# ---------------------------------------------------------------------------
# Inspect British Columbia boundary resources
# ---------------------------------------------------------------------------

bc_boundary_resources = (
    boundary_resource_summary[
        boundary_resource_summary["province"].eq("BC")
    ]
    .sort_values(
        [
            "p50_total_Gt",
            "records",
        ],
        ascending=False,
    )
)

bc_boundary_resources

,province,province_name,canada_overlap_class,PARTNERSHIP,RESOURCE_NAME,records,unique_cells,assessed_records,duplicate_records,p50_total_t,mean_canada_area_pct,min_canada_area_pct,max_canada_area_pct,p50_total_Gt
48,BC,British Columbia,minority_canada,WESTCARB,Whatcom,3,3,3,0,1.247944e+08,21.936536,11.655507,40.868860,0.124794
65,BC,British Columbia,majority_canada,WESTCARB,Whatcom,2,2,2,0,2.508041e+07,73.273168,70.084526,76.461809,0.025080
53,BC,British Columbia,majority_canada,WESTCARB,Hecate Basin,69,69,0,0,NaN,79.097255,50.261187,98.845197,NaN
37,BC,British Columbia,minority_canada,WESTCARB,Hecate Basin,47,47,0,0,NaN,21.048273,1.058106,49.005277,NaN
59,BC,British Columbia,majority_canada,WESTCARB,Queen Charlotte Basin,31,31,0,0,NaN,85.864449,58.002515,98.939811,NaN
42,BC,British Columbia,minority_canada,WESTCARB,Queen Charlotte Basin,21,21,0,0,NaN,22.712526,2.536192,48.909065,NaN
41,BC,British Columbia,minority_canada,WESTCARB,Nanaimo Subbasin,19,19,0,0,NaN,28.160510,1.358844,46.406700,NaN
36,BC,British Columbia,minority_canada,WESTCARB,Comox Subbasin,15,15,0,0,NaN,22.694854,1.265484,45.296573,NaN
58,BC,British Columbia,majority_canada,WESTCARB,Nanaimo Subbasin,14,14,0,0,NaN,74.110956,55.747468,97.430100,NaN
52,BC,British Columbia,majority_canada,WESTCARB,Comox Subbasin,12,12,0,0,NaN,80.418265,54.541285,97.904348,NaN


In [63]:
# ---------------------------------------------------------------------------
# Summarize NATCARB provenance by province and partnership
# ---------------------------------------------------------------------------

province_partnership_summary = (
    saline_canada
    .groupby(
        [
            "province",
            "province_name",
            "PARTNERSHIP",
        ],
        dropna=False,
    )
    .agg(
        records=("source_fid", "count"),
        unique_cells=("COL_ROW", "nunique"),
        resource_count=("RESOURCE_NAME", "nunique"),
        assessed_records=("ASSESSED", lambda s: s.eq(1).sum()),
        duplicate_records=("DUPLICATE", lambda s: s.eq(1).sum()),
        mean_canada_area_pct=("canada_area_pct", "mean"),
        min_canada_area_pct=("canada_area_pct", "min"),
        max_canada_area_pct=("canada_area_pct", "max"),
    )
    .reset_index()
)

# Add non-duplicate P50 totals separately
nonduplicate_partnership_p50 = (
    saline_canada[
        saline_canada["DUPLICATE"].eq(0)
    ]
    .groupby(
        [
            "province",
            "province_name",
            "PARTNERSHIP",
        ],
        dropna=False,
    )["VOL_MED"]
    .apply(
        lambda s: s.sum(min_count=1)
    )
    .rename("nonduplicate_p50_total_t")
    .reset_index()
)

province_partnership_summary = (
    province_partnership_summary
    .merge(
        nonduplicate_partnership_p50,
        on=[
            "province",
            "province_name",
            "PARTNERSHIP",
        ],
        how="left",
    )
)

province_partnership_summary[
    "nonduplicate_p50_total_Gt"
] = (
    province_partnership_summary[
        "nonduplicate_p50_total_t"
    ]
    / 1e9
)

province_partnership_summary.sort_values(
    [
        "province",
        "nonduplicate_p50_total_Gt",
    ],
    ascending=[
        True,
        False,
    ],
    na_position="last",
)

,province,province_name,PARTNERSHIP,records,unique_cells,resource_count,assessed_records,duplicate_records,mean_canada_area_pct,min_canada_area_pct,max_canada_area_pct,nonduplicate_p50_total_t,nonduplicate_p50_total_Gt
1,AB,Alberta,PCOR,12122,4482,7,12092,0,99.836341,1.538058,100.000000,7.627132e+10,76.271316
0,AB,Alberta,BSCSP,259,23,11,259,0,42.476304,1.538058,98.352025,1.614881e+09,1.614881
2,AB,Alberta,WESTCARB,54,54,1,0,6,100.000000,100.000000,100.000000,NaN,NaN
3,BC,British Columbia,PCOR,397,303,5,395,0,100.000000,100.000000,100.000000,1.825146e+09,1.825146
4,BC,British Columbia,WESTCARB,4577,4390,42,6,65,96.837759,0.001282,100.000000,1.498787e+08,0.149879
5,MB,Manitoba,PCOR,1882,1864,3,1882,0,98.571680,0.641136,100.000000,1.361305e+10,13.613051
6,NT,Northwest Territories,PCOR,182,180,2,182,0,100.000000,100.000000,100.000000,1.933599e+08,0.193360
7,NT,Northwest Territories,WESTCARB,14,14,2,0,0,100.000000,100.000000,100.000000,NaN,NaN
8,ON,Ontario,MRCSP,59,21,9,59,0,40.588843,0.019828,97.551955,2.409126e+07,0.024091
10,SK,Saskatchewan,PCOR,6502,3695,7,6502,12,98.836099,0.107200,100.000000,2.854870e+11,285.486992


### Canadian saline provenance

The Canadian-intersecting NATCARB v1502 saline subset is not derived from a
single Canadian assessment.

- PCOR provides the dominant quantitative saline-storage assessment for Alberta,
  Saskatchewan, Manitoba, British Columbia, and the Northwest Territories.
- BSCSP contributes additional cross-border resource coverage in Alberta and
  Saskatchewan, but much of this contribution occurs in boundary cells.
- WESTCARB contributes extensive mapped geological-resource coverage in British
  Columbia, Yukon, and parts of Alberta/NWT, but most WESTCARB records are
  unassessed.
- MRCSP is the sole source of Ontario-intersecting saline records; these occur
  only in boundary cells and should not be interpreted automatically as a
  dedicated Ontario storage assessment.

PARTNERSHIP should therefore be retained as a first-class provenance field in
the standardized dataset.

In [64]:
# ---------------------------------------------------------------------------
# Candidate standardized NATCARB saline schema
# ---------------------------------------------------------------------------

saline_standard_columns = [
    # Source identity / provenance
    "source_fid",
    "COL_ROW",
    "PARTNERSHIP",
    "ARRA_PROJECT",
    "CYCLE_OF_LAST_UPDATE",

    # Geological identity
    "RESOURCE_NAME",
    "BASIN_NAME",

    # Resource extent
    "RSC_AREA_CELL",
    "geometry_area_m2",

    # Capacity estimates
    "VOL_LOW",
    "VOL_MED",
    "VOL_HIGH",

    # Reservoir properties
    "DEPTH_FT",
    "THICKNESS_FT",
    "SALINITY_TDS",
    "PRESSURE_PSI",
    "TEMPERATURE_F",
    "POROSITY_PCT",
    "PERMEABILITY_mD",

    # NATCARB QA / provenance
    "ASSESSED",
    "OVERLAP",
    "DUPLICATE",
    "MED_CALCED",

    # Canadian spatial enrichment
    "province",
    "province_name",
    "province_assignment_method",
    "canada_intersection_area_m2",
    "canada_area_fraction",
    "canada_area_pct",
    "canada_overlap_class",
    "intersects_canada",

    # Geometry
    "geometry",
]

## Saline 10 km reconnaissance conclusions

The exploration of `NATCARB_Saline_10K_v1502` is sufficiently complete to
define how the official NATCARB v1502 saline layer should be interpreted in a
future standardized geological-storage workflow.

This notebook remains a reconnaissance and schema-development exercise.
No storage capacities have been spatially rescaled, and no cross-source
capacity harmonization has yet been applied.

### Source structure

The official NATCARB v1502 saline 10 km layer contains:

- **186,675 source features**
- a native FileGDB feature identifier (`source_fid`) that is unique for all
  186,675 records;
- a nominal 10 km grid identifier (`COL_ROW`);
- geological-resource and provenance attributes;
- low / medium / high storage-resource estimates;
- reservoir-property fields with variable completeness;
- source QA fields describing assessment, overlap, duplication, and the origin
  of the medium estimate.

`source_fid` should be retained as the authoritative identifier for an
individual NATCARB source feature.

`COL_ROW` should not be treated as a record identifier. Multiple geological
resource observations can occur within the same nominal 10 km grid cell.

Within the Canadian-intersecting subset:

- **26,612 source records** were identified;
- these records occupy **14,789 unique `COL_ROW` cells**;
- multiple resource observations can therefore occupy the same nominal grid
  location.

Descriptive fields such as `PARTNERSHIP`, `RESOURCE_NAME`, and `BASIN_NAME`
also do not form a reliable natural primary key. Records were identified that
share the same cell, partnership, basin, resource name, and geometry while
retaining distinct quantitative resource-area and capacity values.

No source records should therefore be deduplicated solely from descriptive or
spatial attributes.

### Geological-resource identity

`RESOURCE_NAME` is the most consistently populated geological descriptor, but
it is not a globally unique geological-resource identifier.

`BASIN_NAME` is sparsely populated and cannot be required as part of a
canonical geological hierarchy. Some records also encode basin-like
information within `RESOURCE_NAME`.

The v1502 saline layer should therefore be interpreted primarily as a
collection of **source resource-cell observations**, rather than as a
normalized database of uniquely identified geological formations, reservoirs,
or basins.

A harmonized geological-resource identifier should not be manufactured from
v1502 fields alone.

### Geometry and resource area

The supplied feature geometry and `Shape_Area` are equivalent for the
Canadian-intersecting subset.

However, `RSC_AREA_CELL` has a different meaning:

- **77.75%** of Canadian-intersecting records have
  `RSC_AREA_CELL == geometry area`;
- **22.16%** have a supplied geometry larger than the reported resource area;
- no records have a reported resource area larger than the supplied geometry.

`RSC_AREA_CELL` must therefore be retained as a first-class source attribute
and must not be reconstructed from polygon geometry.

The supplied geometry represents the spatial feature associated with a NATCARB
grid record, while `RSC_AREA_CELL` represents the reported area of the
geological resource within that nominal cell.

The two quantities should remain distinct in the standardized dataset.

### Capacity semantics and estimate provenance

For assessed records, `VOL_LOW`, `VOL_MED`, and `VOL_HIGH` provide the
low / medium / high storage-resource estimates reported by NATCARB.

The medium estimate does not have uniform provenance. Records include:

- partnership-provided low / medium / high ranges;
- medium estimates calculated from low and high values;
- single estimates copied across the low / medium / high range;
- unassessed records with no storage estimate.

These distinctions should remain explicit.

A future common schema may map these fields to names such as:

- `capacity_p10_t`
- `capacity_p50_t`
- `capacity_p90_t`

where that terminology is appropriate for the source methodology.

However, harmonized field names must **not** be interpreted as evidence that
different storage databases use equivalent uncertainty, capacity-estimation,
or confidence methodologies.

The standardized database should preserve both the reported estimate and its
methodological provenance.

### Assessment, overlap, and duplicate flags

`ASSESSED`, `OVERLAP`, and `DUPLICATE` have distinct meanings and should all be
preserved.

In particular:

- `ASSESSED = 0` represents an unassessed resource and should not be converted
  to zero capacity;
- `OVERLAP = 1` is a source warning and is not by itself a reason to remove a
  record;
- `DUPLICATE = 1` explicitly identifies records NATCARB indicates should not be
  used for quantitative aggregation.

Duplicate records should remain in the standardized data for provenance and
auditability, while downstream quantitative workflows can exclude them.

### Canadian spatial enrichment

Intersecting the NATCARB grid with Canadian provincial and territorial
boundaries identified **26,612 Canadian-intersecting source records**.

A derived `canada_area_pct` field records the fraction of each supplied
NATCARB feature geometry that intersects Canadian provincial or territorial
geometry.

The exploratory overlap classes are:

- `trace_intersection`
- `minority_canada`
- `majority_canada`
- `effectively_canada`

These fields are **spatial QA and jurisdictional-context attributes**.

They should not be interpreted directly as fractions of geological storage
capacity belonging to Canada.

The non-duplicate P50 total associated with all Canadian-intersecting NATCARB
records is approximately **382.076 Gt**.

Of the published capacity associated with these records:

- `effectively_canada`: **374.542 Gt (98.028%)**
- `majority_canada`: **1.860 Gt (0.487%)**
- `minority_canada`: **5.393 Gt (1.412%)**
- `trace_intersection`: **0.281 Gt (0.074%)**

These values classify published NATCARB storage estimates according to the
spatial overlap of their supplied feature geometries with Canadian
provincial/territorial boundaries.

They do **not** represent a physical partition of geological storage capacity
between Canada and the United States.

### Geological capacity versus political jurisdiction

Political boundaries are not physical reservoir boundaries.

A saline formation may be hydraulically connected across provincial,
territorial, or international borders. Injection from a well located in Canada
may create pressure changes, and potentially a CO2 plume, that extend beyond
the political jurisdiction containing the well.

Accordingly:

- `canada_area_pct` should not be multiplied directly by `VOL_LOW`,
  `VOL_MED`, or `VOL_HIGH`;
- a 70% Canadian grid-cell overlap does not imply that 70% of the geological
  storage capacity is physically Canadian;
- a resource crossing the Canada-US border should not be treated as though the
  border acts as an impermeable geological barrier;
- published NATCARB capacity values should remain unchanged in the
  standardized source representation.

This distinction separates three different concepts:

1. **Physical geological storage resource**  
   Capacity associated with the connected geological formation or reservoir.

2. **Jurisdictional attribution**  
   The portion of a resource, storage right, lease, or permitted injection
   activity associated with a political or regulatory jurisdiction.

3. **Model-eligible storage capacity**  
   The capacity that a particular CANOE scenario permits the optimization
   model to access.

These concepts should not be collapsed into a single spatial capacity field.

### Interpretation of Canadian overlap

The Canadian boundary overlay is still valuable.

It identifies:

- resources that are overwhelmingly represented within Canadian terrestrial
  geometry;
- cells that cross the Canada-US border;
- coastal and potentially offshore edge cases;
- records that only minimally intersect Canada;
- candidate locations requiring additional jurisdictional review.

The main downstream question should therefore be:

> Can this storage opportunity reasonably be accessed from an injection
> location permitted by the scenario?

rather than:

> What percentage of this published geological capacity lies mathematically
> inside Canada?

For example, a cell with only a trace Canadian intersection may be unsuitable
as a Canadian injection opportunity even though it belongs to a larger
transboundary reservoir.

Conversely, a resource that is predominantly within Saskatchewan but extends
into Montana should not automatically have its published capacity reduced
according to the political-border fraction.

Any downstream eligibility threshold should therefore operate on injection
location, jurisdiction, storage rights, regulatory assumptions, or explicit
resource classification rather than automatically scaling geological capacity
by `canada_area_pct`.

### Provincial and partnership provenance

The Canadian-intersecting records are assembled from multiple NATCARB regional
partnership assessments rather than from a single Canadian national assessment.

The dominant quantitative provenance is **PCOR**.

Approximate non-duplicate P50 contributions include:

- Alberta: **76.27 Gt** from PCOR;
- Saskatchewan: **285.49 Gt** from PCOR;
- Manitoba: **13.61 Gt** from PCOR;
- Northwest Territories: **0.19 Gt** from PCOR;
- British Columbia: **1.83 Gt** from PCOR.

Additional patterns include:

- BSCSP contributes cross-border assessed resources in Alberta and
  Saskatchewan;
- WESTCARB contributes extensive mapped geological-resource coverage in
  British Columbia, Yukon, and parts of western and northern Canada, but much
  of this coverage is unassessed;
- Ontario contains only **59 MRCSP records across 21 cells**, none of which are
  `effectively_canada`. These should be treated as Canadian-intersecting
  cross-border records rather than automatically interpreted as a dedicated
  Ontario saline-storage assessment;
- Yukon contains mapped resources but no P50 capacity estimates in this
  subset.

`PARTNERSHIP` should therefore remain a first-class provenance attribute.

The regional partnership source helps explain both methodological variation and
why some Canadian-intersecting records are primarily associated with
cross-border US assessments.

### Basin and transboundary-resource interpretation

A geological basin or saline storage formation should not be divided into
separate physical resources simply because it crosses a political boundary.

Where geological evidence supports continuity, the conceptual structure should
remain:

```text
geological basin / storage resource
        |
        +-- Canadian jurisdictional portion
        |
        +-- US jurisdictional portion
        |
        +-- individual source/grid observations